# Transformaciones lineales: núcleo e imagen
### Una casa en 3D, sus planos y su sombra… y lo que pasa con los datos

**Álgebra Lineal · Actividad de autoestudio (no calificable)**

---

### ¿Qué vas a aprender?

Al terminar este cuaderno podrás:

1. Reconocer cuándo una función $T:\mathbb{R}^n\to\mathbb{R}^m$ es una **transformación lineal** y explicar por qué toda transformación lineal se escribe como $T(\mathbf{x})=A\mathbf{x}$.
2. Leer las columnas de $A$ como las imágenes de los vectores canónicos $\mathbf e_1,\mathbf e_2,\mathbf e_3$.
3. Calcular e interpretar geométricamente el **núcleo** $\operatorname{Nu}(T)$ y la **imagen** (o **rango**) $\operatorname{Im}(T)$.
4. Usar el **teorema de la dimensión**: $\dim\operatorname{Nu}(T)+\dim\operatorname{Im}(T)=n$.
5. Explicar qué información se pierde cuando el núcleo no es solo $\{\mathbf 0\}$, con figuras y con datos.

### Cómo usar este cuaderno

* Ejecuta las celdas **en orden** (`Shift + Enter`). Solo se usan `numpy` y `matplotlib`. Funciona en Jupyter, JupyterLab, VS Code o Google Colab.
* 🔮 **Predice antes de ejecutar.** Cuando veas este símbolo, escribe tu predicción (en papel o en una celda) *antes* de correr el código. Equivocarse en una predicción es una de las mejores maneras de aprender.
* 🤔 **Preguntas para pensar.** Son de autoestudio. Intenta responderlas y luego abre la respuesta sugerida (▶).
* ✍️ **Tu turno.** Celdas para modificar y experimentar. Nada se daña: si algo sale mal, deja la celda como estaba y vuelve a ejecutarla.
* No hay nota: el objetivo es entender. Si te equivocas, mejor: ahí está el aprendizaje.

**Duración estimada:** 2 a 3 horas (Actividad 1: unos 90 minutos; Actividad 2: unos 60 minutos).

### Contenido

* **Preparación**
* **Actividad 1. Una casa en 3D**
  1.1 La casa es un conjunto de vectores · 1.2 ¿Qué es una transformación lineal? · 1.3 Galería de transformaciones · 1.4 ¿Qué *no* es lineal? · 1.5 Componer transformaciones · 1.6 Núcleo e imagen · 1.7 Proyectar la casa sobre un plano · 1.8 Resumen y reto
* **Actividad 2. Transformaciones lineales con datos**
  2.1 Colores que se vuelven grises · 2.2 Una tabla de datos de viviendas
* **Síntesis y preguntas de cierre**

## Preparación

Ejecuta la siguiente celda. Contiene las herramientas para **construir y dibujar la casa**. No necesitas entender cada línea de dibujo; lo importante para el curso está en las celdas que siguen, y allí cada cálculo se hace a la vista.

In [ ]:
# =====================================================================
#  CAJA DE HERRAMIENTAS DE DIBUJO (solo numpy y matplotlib)
#  Ejecuta esta celda una vez. No necesitas leer cada línea.
# =====================================================================
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.patches import Polygon, Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

np.set_printoptions(precision=3, suppress=True)          # mostrar números con 3 decimales
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
print("numpy", np.__version__, "· matplotlib", matplotlib.__version__)

# ---------- 1. La casa: vértices (columnas de V) y caras ----------
def construir_casa(altura_techo=2.5, altura_chimenea=None):
    """Devuelve V (3 x N, cada COLUMNA es un vértice), las caras y los detalles (puerta, ventanas)."""
    if altura_chimenea is None:
        altura_chimenea = altura_techo + 0.2
    puntos = []
    def p(x, y, z):                                       # agrega un vértice y devuelve su número
        puntos.append((x, y, z)); return len(puntos) - 1
    a, b, h = 2.0, 3.0, 1.6                              # ancho (x), fondo (y), altura de las paredes (z)
    v000, v100, v110, v010 = p(0, 0, 0), p(a, 0, 0), p(a, b, 0), p(0, b, 0)
    v001, v101, v111, v011 = p(0, 0, h), p(a, 0, h), p(a, b, h), p(0, b, h)
    c0, c1 = p(a/2, 0, altura_techo), p(a/2, b, altura_techo)       # cumbrera del techo
    caras = [
        ([v000, v100, v110, v010], "#d9c9a8"),            # piso
        ([v000, v100, v101, c0, v001], "#f2e3c6"),        # fachada (frente, y = 0)
        ([v010, v110, v111, c1, v011], "#f2e3c6"),        # pared trasera (y = 3)
        ([v000, v010, v011, v001], "#e8d5b0"),            # pared izquierda (x = 0)
        ([v100, v110, v111, v101], "#e8d5b0"),            # pared derecha (x = 2)
        ([v001, c0, c1, v011], "#b5523b"),                # techo, faldón izquierdo
        ([v101, c0, c1, v111], "#c4603f"),                # techo, faldón derecho
    ]
    pendiente = (altura_techo - h) / (a / 2)              # chimenea apoyada en el faldón derecho
    z_techo = lambda x: altura_techo - pendiente * (x - a / 2)
    x0, x1, y0, y1 = 1.35, 1.65, 1.9, 2.3
    ch = [p(x0, y0, z_techo(x0)), p(x1, y0, z_techo(x1)), p(x1, y1, z_techo(x1)), p(x0, y1, z_techo(x0)),
          p(x0, y0, altura_chimenea), p(x1, y0, altura_chimenea), p(x1, y1, altura_chimenea), p(x0, y1, altura_chimenea)]
    for i, j in [(0, 1), (1, 2), (2, 3), (3, 0)]:
        caras.append(([ch[i], ch[j], ch[j + 4], ch[i + 4]], "#8c6d5a"))
    caras.append(([ch[4], ch[5], ch[6], ch[7]], "#5e4a3d"))
    detalles = [([p(0.75, 0, 0), p(1.25, 0, 0), p(1.25, 0, 1.0), p(0.75, 0, 1.0)], "#6b4226")]   # puerta
    for ya, yb in [(0.5, 1.2), (1.8, 2.5)]:                                                    # ventanas
        detalles.append(([p(a, ya, 0.65), p(a, yb, 0.65), p(a, yb, 1.2), p(a, ya, 1.2)], "#7fb3d5"))
    V = np.array(puntos, dtype=float).T
    return V, caras, detalles

def lista_aristas(caras):
    """Pares (i, j) de vértices unidos por un segmento."""
    E = set()
    for idx, _ in caras:
        for k in range(len(idx)):
            i, j = idx[k], idx[(k + 1) % len(idx)]
            E.add((min(i, j), max(i, j)))
    return sorted(E)

V_CASA, CARAS, DETALLES = construir_casa()
ARISTAS = lista_aristas(CARAS + DETALLES)
N_CUERPO = 7                     # las primeras 7 caras son el cuerpo de la casa; las siguientes, la chimenea

def _signo_afuera(V, idx, centro):
    """+1 si el orden de los vértices de la cara da una normal que apunta hacia afuera; -1 si no."""
    q = V[:, idx]
    n = np.cross(q[:, 1] - q[:, 0], q[:, 2] - q[:, 0])
    return 1.0 if n @ (q.mean(axis=1) - centro) > 0 else -1.0

_c_cuerpo, _c_chimenea = V_CASA[:, :10].mean(axis=1), V_CASA[:, 10:18].mean(axis=1)
ORIENT = [_signo_afuera(V_CASA, idx, _c_cuerpo if i < N_CUERPO else _c_chimenea) for i, (idx, _) in enumerate(CARAS)]
ORIENT_DET = [_signo_afuera(V_CASA, idx, _c_cuerpo) for idx, _ in DETALLES]

def _mira(V, idx, orient, v, signo=1.0):
    """¿La cara (vértices idx de V) mira hacia el observador que está en la dirección v?"""
    q = V[:, idx]
    return np.cross(q[:, 1] - q[:, 0], q[:, 2] - q[:, 0]) @ v * orient * signo > 1e-12

# ---------- 2. Dibujo en 3D ----------
COLORES_BASE = ["#d62728", "#2ca02c", "#1f77b4"]        # e1 rojo, e2 verde, e3 azul
COLOR_NUCLEO, COLOR_IMAGEN = "#c0392b", "#2e86c1"

def limites(objetos, margen=0.35, minimo=1.6):
    """Caja [lo, hi] que contiene los puntos dados (matrices 3 x k) y el origen."""
    P = np.hstack([np.asarray(o, float).reshape(3, -1) for o in objetos])
    lo = np.minimum(P.min(axis=1), 0) - margen
    hi = np.maximum(P.max(axis=1), 0) + margen
    centro, lado = (lo + hi) / 2, np.maximum(hi - lo, minimo)
    return centro - lado / 2, centro + lado / 2

def preparar_ejes(ax, lo, hi, titulo="", elev=22, azim=-58):
    ax.set_proj_type("ortho")
    ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
    ax.set_box_aspect(tuple(hi - lo))                     # la misma escala en x, y, z
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.tick_params(labelsize=7)
    for eje in (ax.xaxis, ax.yaxis, ax.zaxis):
        eje.set_major_locator(MaxNLocator(5))
    ax.set_title(titulo)
    ax.computed_zorder = False                            # dibujar en el orden en que se agregan los objetos
    for eje in (ax.xaxis, ax.yaxis, ax.zaxis):
        eje.pane.set_facecolor((1, 1, 1, 0)); eje.pane.set_edgecolor("0.85")
    for k in range(3):                                    # ejes coordenados que pasan por el origen
        s = np.zeros((3, 2)); s[k] = [lo[k], hi[k]]
        ax.plot(*s, color="0.6", lw=0.8, zorder=0.2)
    ax.scatter([0], [0], [0], color="k", s=14, zorder=9)

def dibujar_casa(ax, V, alpha=0.85, color=None, lw=0.9, zorder=1.0):
    """Dibuja la casa cuyos vértices son las columnas de V (3 x N)."""
    el, az = np.radians(ax.elev), np.radians(ax.azim)
    v = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])   # hacia el observador
    vol = np.cross(V[:, 1] - V[:, 0], V[:, 3] - V[:, 0]) @ (V[:, 4] - V[:, 0])    # orientación de la casa
    plana = abs(vol) < 1e-9 * max(1.0, np.abs(V).max()) ** 3
    signo = np.sign(vol)
    # la chimenea va encima del techo si el faldón donde se apoya (cara 6) mira al observador
    chim_arriba = plana or _mira(V, CARAS[6][0], ORIENT[6], v, signo)
    detalles = [d for i, d in enumerate(DETALLES) if plana or _mira(V, d[0], ORIENT_DET[i], v, signo)]
    grupos = [(CARAS[:N_CUERPO], alpha, zorder),
              (CARAS[N_CUERPO:], alpha, zorder + (0.05 if chim_arriba else -0.05)),
              (detalles, min(1.0, alpha + 0.1), zorder + 0.1)]
    for grupo, a, z in grupos:
        if not grupo:
            continue
        polis = [V[:, idx].T for idx, _ in grupo]
        caras = [to_rgba(color or c, a) for _, c in grupo]
        col = Poly3DCollection(polis, facecolors=caras, edgecolors=to_rgba("0.15", min(1.0, a + 0.25)), linewidths=lw)
        col.set_zorder(z)
        ax.add_collection3d(col)

def dibujar_base(ax, B, nombres=("e₁", "e₂", "e₃")):
    """Flechas desde el origen hasta cada columna de B."""
    for j in range(B.shape[1]):
        v = B[:, j]
        if np.linalg.norm(v) < 1e-9:
            ax.text(0.15, 0.15, -0.3 - 0.3 * j, f"{nombres[j]} = 0", color=COLORES_BASE[j], fontsize=9, zorder=10)
            continue
        ax.quiver(0, 0, 0, *v, color=COLORES_BASE[j], linewidth=2.2, arrow_length_ratio=0.15, zorder=8)
        ax.text(*(v * 1.15), nombres[j], color=COLORES_BASE[j], fontsize=10, fontweight="bold", zorder=10)

def comparar(A, titulo="T(casa)", elev=22, azim=-58, base=True):
    """Izquierda: la casa original. Derecha: la casa transformada W = A V (misma escala)."""
    A = np.asarray(A, float)
    W = A @ V_CASA                                        # ¡toda la transformación es este producto!
    lo, hi = limites([V_CASA, W, np.eye(3), A])
    fig = plt.figure(figsize=(11, 4.7))
    for k, (M, B, t, nom) in enumerate([(V_CASA, np.eye(3), "Casa original", ("e₁", "e₂", "e₃")),
                                        (W, A, titulo, ("T(e₁)", "T(e₂)", "T(e₃)"))]):
        ax = fig.add_subplot(1, 2, k + 1, projection="3d")
        preparar_ejes(ax, lo, hi, t, elev, azim)
        if k == 1:
            dibujar_casa(ax, V_CASA, alpha=0.07, color="0.5", lw=0.3)   # la original, tenue
        dibujar_casa(ax, M)
        if base:
            dibujar_base(ax, B, nom)
    plt.tight_layout(); plt.show()

def comparar_varias(lista, elev=22, azim=-58):
    """lista = [(A, título), ...]: la casa transformada por cada A, todas con la misma escala."""
    Ws = [np.asarray(A, float) @ V_CASA for A, _ in lista]
    lo, hi = limites([V_CASA] + Ws)
    fig = plt.figure(figsize=(4.4 * len(lista), 4.5))
    for k, ((A, t), W) in enumerate(zip(lista, Ws)):
        ax = fig.add_subplot(1, len(lista), k + 1, projection="3d")
        preparar_ejes(ax, lo, hi, t, elev, azim)
        dibujar_casa(ax, V_CASA, alpha=0.07, color="0.5", lw=0.3)
        dibujar_casa(ax, W)
    plt.tight_layout(); plt.show()

def ver_casa(V=None, elev=22, azim=-58, titulo="La casa"):
    V = V_CASA if V is None else V
    lo, hi = limites([V, np.eye(3)])
    fig = plt.figure(figsize=(5.5, 5)); ax = fig.add_subplot(projection="3d")
    preparar_ejes(ax, lo, hi, titulo, elev, azim)
    dibujar_casa(ax, V); dibujar_base(ax, np.eye(3))
    plt.show()

def dibujar_subespacio(ax, B, lo, hi, color, alpha=0.18, zorder=0.5):
    """Dibuja el subespacio generado por las columnas de B (una recta o un plano), recortado a la caja."""
    B = np.asarray(B, float).reshape(3, -1)
    if B.shape[1] == 1:                                   # recta: t·d
        d = B[:, 0] / np.linalg.norm(B[:, 0])
        t0, t1 = -np.inf, np.inf
        for i in range(3):
            if abs(d[i]) > 1e-12:
                a, b = sorted((lo[i] / d[i], hi[i] / d[i]))
                t0, t1 = max(t0, a), min(t1, b)
        ax.plot(*np.outer(d, [t0, t1]), color=color, lw=2.4, ls="--", zorder=zorder + 5)
    elif B.shape[1] == 2:                                 # plano n·x = 0, cortado por las aristas de la caja
        n = np.cross(B[:, 0], B[:, 1]); n = n / np.linalg.norm(n)
        C = np.array([[x, y, z] for x in (lo[0], hi[0]) for y in (lo[1], hi[1]) for z in (lo[2], hi[2])])
        pts = []
        for i in range(8):
            for j in range(i + 1, 8):
                if np.sum(C[i] != C[j]) != 1:
                    continue
                fi, fj = n @ C[i], n @ C[j]
                if abs(fi) < 1e-12: pts.append(C[i])
                if abs(fj) < 1e-12: pts.append(C[j])
                if fi * fj < 0: pts.append(C[i] + (C[j] - C[i]) * fi / (fi - fj))
        pts = np.unique(np.round(np.array(pts), 9), axis=0)
        if len(pts) >= 3:
            c = pts.mean(axis=0)
            u = (pts[0] - c) / np.linalg.norm(pts[0] - c); w = np.cross(n, u)
            orden = np.argsort(np.arctan2((pts - c) @ w, (pts - c) @ u))
            pol = Poly3DCollection([pts[orden]], facecolors=[to_rgba(color, alpha)],
                                   edgecolors=[to_rgba(color, 0.5)], linewidths=0.8)
            pol.set_zorder(zorder); ax.add_collection3d(pol)

def dibujar_rayos(ax, V, W, color="0.45"):
    """Segmentos punteados de cada vértice x a su imagen T(x)."""
    for k in range(V.shape[1]):
        ax.plot(*np.c_[V[:, k], W[:, k]], color=color, lw=0.6, ls=":", zorder=0.8)

def leyenda_nucleo_imagen(ax, rayos=True):
    h = [Patch(color=COLOR_IMAGEN, alpha=0.35, label="Im(T)"),
         Line2D([], [], color=COLOR_NUCLEO, ls="--", lw=2, label="Nu(T)")]
    if rayos:
        h.append(Line2D([], [], color="0.45", ls=":", label="de x a T(x)"))
    ax.legend(handles=h, loc="upper left", fontsize=8)

# ---------- 3. Dibujo en 2D (planos de arquitecto y fotos) ----------
def dibujar_2d(ax, P2, V3=None, v=None, titulo=None, color=None, alpha=1.0, contorno=False, ls="-", bordes=True):
    """Dibuja una vista plana de la casa. P2 (2 x N) son las coordenadas en el papel; V3, los vértices en 3D.
       Si se da la dirección v hacia el observador, solo se pintan las caras que miran hacia él."""
    V3 = V_CASA if V3 is None else V3
    if v is None:                                         # sin observador: todas las caras (siluetas, contornos)
        cuerpo, chimenea, detalles, chim_primero = CARAS[:N_CUERPO], CARAS[N_CUERPO:], DETALLES, False
    else:
        cuerpo = [f for i, f in enumerate(CARAS[:N_CUERPO]) if _mira(V3, f[0], ORIENT[i], v)]
        chimenea = [f for i, f in enumerate(CARAS[N_CUERPO:]) if _mira(V3, f[0], ORIENT[N_CUERPO + i], v)]
        detalles = [d for i, d in enumerate(DETALLES) if _mira(V3, d[0], ORIENT_DET[i], v)]
        chim_primero = not _mira(V3, CARAS[6][0], ORIENT[6], v)   # chimenea detrás del techo
    estilo = lambda c: dict(fc="none" if contorno else to_rgba(color or c, alpha),
                            ec="0.15" if (bordes or contorno) else "none", lw=0.8, ls=ls)
    for idx, c in (chimenea + cuerpo if chim_primero else cuerpo + chimenea) + detalles:
        ax.add_patch(Polygon(P2[:, idx].T, closed=True, **estilo(c)))
    ax.set_aspect("equal"); ax.autoscale_view(); ax.margins(0.08)
    ax.grid(alpha=0.25)
    if titulo:
        ax.set_title(titulo)

print(f"Listo. La casa tiene {V_CASA.shape[1]} vértices, {len(CARAS)} caras y {len(ARISTAS)} aristas.")

---
# Actividad 1. Una casa en 3D

## 1.1 La casa es un conjunto de vectores

Para la computadora, la casa es una lista de puntos de $\mathbb{R}^3$ (los **vértices**) más la información de cuáles puntos se unen para formar paredes, techo, puerta y ventanas. Guardamos los vértices como **columnas** de una matriz $V$ de tamaño $3\times N$:

$$
V=\begin{bmatrix} | & | & & | \\ \mathbf v_1 & \mathbf v_2 & \cdots & \mathbf v_N \\ | & | & & | \end{bmatrix}.
$$

La casa mide 2 de ancho (eje $x$) y 3 de fondo (eje $y$); las paredes miden 1.6 de alto y el techo llega a 2.5. Una de sus esquinas está en el origen. Las flechas roja, verde y azul son los vectores canónicos $\mathbf e_1,\mathbf e_2,\mathbf e_3$.

In [ ]:
print("Tamaño de V:", V_CASA.shape)
print("Los primeros 5 vértices (cada columna es un punto (x, y, z)):")
print(V_CASA[:, :5])

lo, hi = limites([V_CASA, np.eye(3)])
fig = plt.figure(figsize=(11, 4.8))
for k, (elev, azim) in enumerate([(22, -58), (25, 125)]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    preparar_ejes(ax, lo, hi, f"Vista desde elev = {elev}°, azim = {azim}°", elev, azim)
    dibujar_casa(ax, V_CASA)
    dibujar_base(ax, np.eye(3))
plt.tight_layout(); plt.show()

✍️ **Tu turno.** Cambia el punto de vista (`elev` es la altura del observador en grados; `azim`, el giro alrededor del eje $z$).

In [ ]:
ver_casa(elev=40, azim=-20, titulo="Mi vista de la casa")

### 🤔 Preguntas para pensar (1.1)

a) ¿Cuántas columnas tiene $V$? ¿Qué representa cada columna? ¿Y cada una de sus tres filas?

b) Busca en la salida el vértice que está en el origen. Sin ejecutar nada: ¿a dónde irá esa esquina después de **cualquier** transformación lineal? Guarda tu respuesta; la comprobarás en la sección 1.2.

c) Para dibujar la casa también necesitamos saber qué vértices se unen con segmentos. Cuando transformemos la casa, ¿crees que esa lista de conexiones tendrá que cambiar? Piensa una razón antes de seguir.

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) $V$ tiene 30 columnas, una por vértice. La primera fila contiene las coordenadas $x$ de todos los vértices; la segunda, las $y$; la tercera, las $z$.

b) Al origen. Toda transformación lineal cumple $T(\mathbf 0)=\mathbf 0$, porque $T(\mathbf 0)=T(0\cdot\mathbf 0)=0\cdot T(\mathbf 0)=\mathbf 0$. Por eso, en todas las figuras de este cuaderno, la esquina de la casa que está en el origen nunca se mueve.

c) No cambia. Una transformación lineal lleva segmentos en segmentos (lo verás en 1.2), así que basta transformar los vértices y volver a unirlos con la misma lista de conexiones.

</details>

## 1.2 ¿Qué es una transformación lineal?

Una función $T:\mathbb{R}^n\to\mathbb{R}^m$ es **lineal** si para todos los vectores $\mathbf u,\mathbf v$ y todo escalar $c$:

1. $T(\mathbf u+\mathbf v)=T(\mathbf u)+T(\mathbf v)$ (respeta sumas),
2. $T(c\,\mathbf u)=c\,T(\mathbf u)$ (respeta múltiplos).

**Toda transformación lineal es una multiplicación por una matriz.** Escribe $\mathbf x=x_1\mathbf e_1+x_2\mathbf e_2+x_3\mathbf e_3$ y usa las dos propiedades:

$$
T(\mathbf x)=x_1T(\mathbf e_1)+x_2T(\mathbf e_2)+x_3T(\mathbf e_3)
=\underbrace{\begin{bmatrix} | & | & | \\ T(\mathbf e_1) & T(\mathbf e_2) & T(\mathbf e_3)\\ | & | & | \end{bmatrix}}_{A}\begin{bmatrix}x_1\\x_2\\x_3\end{bmatrix}=A\mathbf x .
$$

> **Las columnas de $A$ son las imágenes de $\mathbf e_1,\mathbf e_2,\mathbf e_3$.** Para conocer $T$ en todo el espacio basta saber qué le hace a tres vectores. En las figuras dibujamos esas tres flechas: $T(\mathbf e_1)$ en rojo, $T(\mathbf e_2)$ en verde y $T(\mathbf e_3)$ en azul.

**¿Por qué basta transformar los vértices?** Los puntos del segmento que une $\mathbf u$ con $\mathbf v$ son $(1-t)\mathbf u+t\mathbf v$, con $0\le t\le 1$. Por linealidad,

$$
T\big((1-t)\mathbf u+t\mathbf v\big)=(1-t)\,T(\mathbf u)+t\,T(\mathbf v),
$$

que es el punto correspondiente del segmento que une $T(\mathbf u)$ con $T(\mathbf v)$. Así, **la casa transformada completa se obtiene con un solo producto de matrices**: $W=AV$, porque la columna $k$ de $W$ es $A\mathbf v_k$.

In [ ]:
# Una matriz cualquiera y la comprobación de que sus columnas son T(e1), T(e2), T(e3)
A = np.array([[2.0, 0.0, 0.0],
              [0.0, 1.0, 0.0],
              [0.0, 0.0, 0.5]])
e1, e2, e3 = np.eye(3)                     # las filas de la identidad son e1, e2, e3
print("A e1 =", A @ e1, "  ← primera columna de A")
print("A e2 =", A @ e2, "  ← segunda columna de A")
print("A e3 =", A @ e3, "  ← tercera columna de A")

x = np.array([1.0, 3.0, 2.0])
print("\nA x                          =", A @ x)
print("1·T(e1) + 3·T(e2) + 2·T(e3)  =", 1 * (A @ e1) + 3 * (A @ e2) + 2 * (A @ e3))

W = A @ V_CASA                             # la casa transformada: una sola multiplicación
print("\nVértice del origen: antes", V_CASA[:, 0], "→ después", W[:, 0])
comparar(A, "T(x) = A x")

### 🤔 Preguntas para pensar (1.2)

a) Si $T(\mathbf e_1)=(2,0,0)$, $T(\mathbf e_2)=(0,1,0)$ y $T(\mathbf e_3)=(0,0,\tfrac12)$, escribe la matriz $A$. Describe con palabras cómo queda la casa y compáralo con la figura.

b) Explica con tus palabras por qué basta transformar los 30 vértices y no cada punto de cada pared.

c) Si $T$ es lineal y $T(\mathbf u)=(1,2,0)$, ¿cuánto vale $T(-3\mathbf u)$? ¿Y $T(\mathbf 0)$?

d) Una matriz de tamaño $2\times 3$, ¿define una transformación lineal de dónde a dónde? ¿Y una de $3\times 2$?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) $A=\begin{bmatrix}2&0&0\\0&1&0\\0&0&\tfrac12\end{bmatrix}$, exactamente la matriz de la celda: sus columnas son las imágenes dadas. La casa queda el doble de ancha (en $x$), igual de profunda (en $y$) y con la mitad de la altura (en $z$).

b) Cada punto de una arista es de la forma $(1-t)\mathbf u+t\mathbf v$ y su imagen es $(1-t)T(\mathbf u)+tT(\mathbf v)$: el punto correspondiente del segmento entre las imágenes. Lo mismo vale para los puntos de una pared, que son combinaciones de sus vértices con coeficientes no negativos que suman 1. Por eso, transformar los vértices y unirlos con las mismas conexiones produce la casa transformada exacta, no una aproximación.

c) $T(-3\mathbf u)=-3\,T(\mathbf u)=(-3,-6,0)$ y $T(\mathbf 0)=\mathbf 0$.

d) Una matriz $2\times 3$ multiplica vectores de 3 componentes y produce vectores de 2 componentes: $T:\mathbb{R}^3\to\mathbb{R}^2$. Una de $3\times 2$ define $T:\mathbb{R}^2\to\mathbb{R}^3$. El número de **columnas** es la dimensión del dominio y el de **filas**, la del codominio.

</details>

## 1.3 Galería de transformaciones

Cada transformación de esta galería es una matriz $3\times 3$. Antes de ejecutar cada celda, mira la matriz y **predice** 🔮 qué le pasará a la casa. Después compara con la figura y con las flechas $T(\mathbf e_j)$, que son las columnas de la matriz.

### a) Escalamiento
$$
S=\begin{bmatrix}1.5&0&0\\0&1&0\\0&0&0.6\end{bmatrix}
$$
🔮 ¿Qué medidas de la casa cambian y cuáles no?

In [ ]:
S = np.diag([1.5, 1.0, 0.6])
print("S =\n", S)
comparar(S, "Escalamiento S")

### b) Rotación alrededor del eje $z$
$$
R_\theta=\begin{bmatrix}\cos\theta&-\sin\theta&0\\ \sin\theta&\cos\theta&0\\0&0&1\end{bmatrix},\qquad \theta=60^\circ .
$$
🔮 ¿Hacia dónde girará la casa? Recuerda que el eje de giro pasa por el origen, que es una esquina de la casa, no su centro.

In [ ]:
def rotacion_z(grados):
    """Matriz de rotación de 'grados' alrededor del eje z (sentido antihorario visto desde arriba)."""
    t = np.radians(grados)
    return np.array([[np.cos(t), -np.sin(t), 0.0],
                     [np.sin(t),  np.cos(t), 0.0],
                     [0.0,        0.0,       1.0]])

R60 = rotacion_z(60)
print("R60 =\n", R60)
comparar(R60, "Rotación de 60° alrededor de z")

### c) Reflexión (espejo en el plano $xz$)
$$
M=\begin{bmatrix}1&0&0\\0&-1&0\\0&0&1\end{bmatrix}
$$
🔮 ¿Dónde quedará la casa? ¿Qué flecha cambia de sentido?

In [ ]:
M = np.diag([1.0, -1.0, 1.0])
print("M =\n", M)
comparar(M, "Reflexión en el plano xz", azim=-35)

### d) Cizalla (la casa se inclina)
$$
C=\begin{bmatrix}1&0&0.6\\0&1&0\\0&0&1\end{bmatrix},\qquad C(x,y,z)=(x+0.6z,\ y,\ z).
$$
🔮 ¿Qué puntos de la casa no se mueven? ¿Cambia la altura de algún punto?

In [ ]:
C = np.array([[1.0, 0.0, 0.6],
              [0.0, 1.0, 0.0],
              [0.0, 0.0, 1.0]])
print("C =\n", C)
comparar(C, "Cizalla C")

### 🤔 Preguntas para pensar (1.3)

a) En la rotación, ¿cambia el tamaño de la casa? Compruébalo calculando las longitudes de las columnas de $R_{60}$ y los productos punto entre ellas. (✍️ Usa la celda de abajo.)

b) La reflexión deja algunos vectores exactamente donde estaban y a otros les invierte el sentido. ¿Cuáles son unos y cuáles los otros?

c) En la cizalla, ¿qué puntos de la casa quedan fijos? ¿Cambia la altura de algún punto? ¿Por qué las paredes siguen siendo planas?

d) ¿Alguna de las cuatro transformaciones "aplasta" la casa, es decir, la deja contenida en un plano o en una recta? ¿Cómo lo reconoces mirando las tres flechas $T(\mathbf e_1),T(\mathbf e_2),T(\mathbf e_3)$?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) No cambia de tamaño. Las columnas de $R_{60}$ tienen norma 1 y son perpendiculares entre sí (en la celda, `R60.T @ R60` da la identidad), así que la rotación conserva longitudes y ángulos: la casa solo gira. Si la casa "se mueve de lugar" es porque el eje de giro pasa por una esquina y no por el centro.

b) Los vectores del plano $xz$ (los que tienen $y=0$) quedan fijos: $M(x,0,z)=(x,0,z)$. Los vectores en la dirección del eje $y$ cambian de sentido: $M(0,y,0)=(0,-y,0)$. En cualquier otro vector solo cambia el signo de la componente $y$.

c) $C(x,y,z)=(x+0.6z,\,y,\,z)$. Los puntos con $z=0$ (el piso) quedan fijos y los demás se desplazan en $x$ en proporción a su altura. Ninguna altura cambia, porque la tercera fila de $C$ es $(0,0,1)$. Las paredes siguen siendo planas porque $C$ es lineal: lleva segmentos en segmentos y planos en planos.

d) Ninguna. En las cuatro, las tres flechas no están en un mismo plano (son linealmente independientes), así que la casa conserva su volumen. Si dos o tres de las flechas quedaran en un mismo plano, la casa se aplastaría. En la sección 1.6 diremos esto con precisión usando el núcleo.

</details>

In [ ]:
# ✍️ Comprobación de la pregunta a)
print("Longitudes de las columnas de R60:", np.linalg.norm(R60, axis=0))
print("R60ᵀ R60 =\n", np.round(R60.T @ R60, 12) + 0.0)

✍️ **Tu turno.** Escribe tu propia matriz y mira qué le hace a la casa. Algunas ideas:

1. `[[1, 0.5, 0], [0, 1, 0], [0, 0, 1]]`
2. `[[-1, 0, 0], [0, -1, 0], [0, 0, 1]]`: ¿a qué rotación equivale?
3. Una matriz con dos columnas iguales, por ejemplo `[[1, 1, 0], [0, 0, 0], [0, 0, 1]]`. ¿Qué le pasa a la casa? Guarda esta observación para la sección 1.6.

In [ ]:
A_mia = np.array([[1.0, 0.0, 0.0],
                  [0.0, 1.0, 0.0],
                  [0.0, 0.0, 1.0]])
comparar(A_mia, "Mi transformación")

## 1.4 ¿Qué *no* es una transformación lineal?

Veamos dos funciones que también mueven la casa:

* **Traslación:** $f(\mathbf x)=\mathbf x+\mathbf b$, con $\mathbf b=(1.5,\ 1,\ 0.5)$.
* **Torsión:** cada punto gira alrededor del eje $z$ un ángulo que depende de su altura, $f(\mathbf x)=R_{\theta(\mathbf x)}\,\mathbf x$ con $\theta(\mathbf x)=0.25\,x_3$ radianes.

🔮 ¿Cuál de las dos crees que es lineal? ¿Alguna?

La función `es_lineal` prueba las dos propiedades de la definición con muchos vectores elegidos al azar. Léela: es corta.

In [ ]:
def es_lineal(f, n=3, pruebas=200, semilla=0):
    """Prueba numérica: ¿f(u+v) = f(u)+f(v) y f(cu) = c f(u) para vectores al azar?
       Si encuentra una falla, muestra el contraejemplo. Pasar la prueba NO es una demostración."""
    rng = np.random.default_rng(semilla)
    for _ in range(pruebas):
        u, v, c = rng.normal(size=n), rng.normal(size=n), rng.normal()
        if not np.allclose(f(u + v), f(u) + f(v)):
            print("  ✗ No es lineal: falla f(u+v) = f(u) + f(v)")
            print("    con u =", u, " v =", v)
            print("    f(u+v) =", f(u + v), "   f(u)+f(v) =", f(u) + f(v))
            return
        if not np.allclose(f(c * u), c * f(u)):
            print(f"  ✗ No es lineal: falla f(cu) = c f(u) con c = {c:.3f}")
            return
    print(f"  ✓ Pasó {pruebas} pruebas al azar: parece lineal (una prueba numérica no es una demostración).")

b = np.array([1.5, 1.0, 0.5])
traslacion = lambda x: x + b
torsion = lambda x: rotacion_z(np.degrees(0.25 * x[2])) @ x
rotacion = lambda x: R60 @ x

for nombre, f in [("Rotación R60", rotacion), ("Traslación", traslacion), ("Torsión", torsion)]:
    print(nombre, " → f(0) =", f(np.zeros(3)))
    es_lineal(f)

In [ ]:
def dibujar_casa_funcion(ax, f, n=30, color="0.15"):
    """Para funciones NO lineales: transforma muchos puntos de cada arista y dibuja las curvas resultantes."""
    t = np.linspace(0, 1, n)
    for i, j in ARISTAS:
        P = np.outer(V_CASA[:, i], 1 - t) + np.outer(V_CASA[:, j], t)      # puntos del segmento
        Q = np.array([f(P[:, k]) for k in range(n)]).T
        ax.plot(*Q, color=color, lw=1.1, zorder=2)

V_tras = V_CASA + b[:, None]                       # la traslación se aplica sumando b a cada columna
V_tors = np.array([torsion(V_CASA[:, k]) for k in range(V_CASA.shape[1])]).T
lo, hi = limites([V_CASA, V_tras, V_tors])
fig = plt.figure(figsize=(12, 4.6))
ax = fig.add_subplot(1, 2, 1, projection="3d")
preparar_ejes(ax, lo, hi, "Traslación: f(x) = x + b")
dibujar_casa(ax, V_CASA, alpha=0.1, color="0.5", lw=0.3)
dibujar_casa(ax, V_tras)
ax.quiver(0, 0, 0, *b, color="k", linewidth=2, arrow_length_ratio=0.15, zorder=8)
ax.text(*(b * 1.1), "b", fontweight="bold", zorder=10)
ax = fig.add_subplot(1, 2, 2, projection="3d")
preparar_ejes(ax, lo, hi, "Torsión: el ángulo de giro crece con la altura")
dibujar_casa(ax, V_CASA, alpha=0.1, color="0.5", lw=0.3)
dibujar_casa_funcion(ax, torsion, color="#b5523b")
plt.tight_layout(); plt.show()

### 🤔 Preguntas para pensar (1.4)

a) La traslación no deforma la casa: conserva longitudes, ángulos y rectas. Entonces, ¿por qué no es lineal? Da dos razones.

b) ¿Se puede escribir la traslación como $f(\mathbf x)=A\mathbf x$ para alguna matriz $A$ de $3\times 3$? Justifica.

c) La torsión convierte aristas rectas en curvas. Explica por qué **ninguna** transformación lineal puede convertir un segmento en una curva.

d) La torsión cumple $f(\mathbf 0)=\mathbf 0$ y aun así no es lineal. ¿Qué te dice esto sobre la condición $T(\mathbf 0)=\mathbf 0$? Da otro ejemplo de una función con $f(\mathbf 0)=\mathbf 0$ que no sea lineal.

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) Primera razón: $f(\mathbf 0)=\mathbf b\neq\mathbf 0$, y toda transformación lineal manda $\mathbf 0$ a $\mathbf 0$. Segunda razón: $f(\mathbf u+\mathbf v)=\mathbf u+\mathbf v+\mathbf b$, mientras que $f(\mathbf u)+f(\mathbf v)=\mathbf u+\mathbf v+2\mathbf b$, y no son iguales. Conservar la forma no es lo mismo que ser lineal.

b) No. Para cualquier matriz, $A\mathbf 0=\mathbf 0$, y la traslación manda $\mathbf 0$ a $\mathbf b$. (Como curiosidad: en computación gráfica se usan *coordenadas homogéneas*, vectores de $\mathbb{R}^4$, para que las traslaciones sí se escriban como un producto por una matriz $4\times 4$.)

c) Si $T$ es lineal, el segmento $(1-t)\mathbf u+t\mathbf v$ se transforma en $(1-t)T(\mathbf u)+tT(\mathbf v)$, que es otra vez un segmento (o un punto, si $T(\mathbf u)=T(\mathbf v)$). Nunca puede salir una curva.

d) $T(\mathbf 0)=\mathbf 0$ es una condición **necesaria** pero **no suficiente**: si falla, $T$ no es lineal; si se cumple, todavía hay que verificar las dos propiedades. Otro ejemplo: $f(x,y,z)=(x^2,\,y,\,z)$ cumple $f(\mathbf 0)=\mathbf 0$, pero $f(2\mathbf e_1)=(4,0,0)\neq 2f(\mathbf e_1)=(2,0,0)$.

</details>

## 1.5 Componer transformaciones

Si primero aplicamos $A$ y luego $B$:

$$
\mathbf x\ \longmapsto\ A\mathbf x\ \longmapsto\ B(A\mathbf x)=(BA)\,\mathbf x .
$$

La composición también es lineal y su matriz es el producto $BA$. Ojo con el orden: **la que se aplica primero queda a la derecha.**

🔮 ¿Da lo mismo rotar $90^\circ$ y luego aplicar la cizalla $C$, que aplicar primero la cizalla y luego rotar?

In [ ]:
R90 = rotacion_z(90)
primero_R_luego_C = C @ R90          # se aplica primero R90 (es la que está a la derecha)
primero_C_luego_R = R90 @ C
print("C·R90 =\n", primero_R_luego_C)
print("R90·C =\n", primero_C_luego_R)
print("¿Son iguales?", np.allclose(primero_R_luego_C, primero_C_luego_R))
comparar_varias([(np.eye(3), "Original"),
                 (primero_R_luego_C, "Primero R90, luego C   (C·R90)"),
                 (primero_C_luego_R, "Primero C, luego R90   (R90·C)")])

### 🤔 Preguntas para pensar (1.5)

a) Al componer, ¿por qué la matriz que se aplica primero queda a la derecha del producto?

b) Busca en la galería dos transformaciones que sí **conmuten** ($AB=BA$) y compruébalo con la celda de abajo. ¿Por qué crees que conmutan?

c) ¿Qué matriz "deshace" la rotación $R_{60}$? Compruébalo multiplicando.

d) ¿Se puede deshacer la cizalla $C$? Propón una matriz y compruébala.

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) Porque $(BA)\mathbf x=B(A\mathbf x)$: el vector $\mathbf x$ "se encuentra" primero con la matriz que tiene más cerca, la de la derecha.

b) Por ejemplo $S$ y $M$: las dos son diagonales, y escalar cada eje por separado no depende del orden. También conmutan dos rotaciones alrededor del mismo eje, como $R_{60}$ y $R_{30}$: girar $60^\circ$ y luego $30^\circ$ es lo mismo que al revés. En cambio, $C$ y $R_{90}$ no conmutan, como se ve en la figura.

c) La rotación de $-60^\circ$: `rotacion_z(-60) @ R60` da la identidad. (Para rotaciones, la inversa también coincide con la transpuesta: $R_{60}^{T}R_{60}=I$.)

d) Sí: $C^{-1}=\begin{bmatrix}1&0&-0.6\\0&1&0\\0&0&1\end{bmatrix}$ devuelve cada punto a su lugar restando $0.6z$ a la coordenada $x$. Más adelante veremos que se puede deshacer porque su núcleo es $\{\mathbf 0\}$.

</details>

In [ ]:
# ✍️ Comprobaciones de las preguntas b), c) y d)
print("¿S y M conmutan?          ", np.allclose(S @ M, M @ S))
print("¿R60 y R30 conmutan?      ", np.allclose(R60 @ rotacion_z(30), rotacion_z(30) @ R60))
print("rotacion_z(-60) · R60 =\n", np.round(rotacion_z(-60) @ R60, 12) + 0.0)
C_inv = np.array([[1.0, 0.0, -0.6], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
print("C_inv · C =\n", C_inv @ C)

## 1.6 Núcleo e imagen

Sea $T:\mathbb{R}^n\to\mathbb{R}^m$, $T(\mathbf x)=A\mathbf x$.

* **Núcleo:** $\operatorname{Nu}(T)=\{\mathbf x\in\mathbb{R}^n:\ T(\mathbf x)=\mathbf 0\}$. Es todo lo que $T$ manda al origen, *lo que $T$ no ve*. Es el conjunto solución del sistema homogéneo $A\mathbf x=\mathbf 0$ y es un subespacio del **dominio** $\mathbb{R}^n$. (En algunos textos se escribe $\ker T$, del inglés *kernel*.)
* **Imagen** (también llamada **rango** o recorrido): $\operatorname{Im}(T)=\{T(\mathbf x):\ \mathbf x\in\mathbb{R}^n\}$. Es todo lo que $T$ puede producir. Es un subespacio del **codominio** $\mathbb{R}^m$ y es el espacio generado por las columnas de $A$:
$$\operatorname{Im}(T)=\operatorname{gen}\{T(\mathbf e_1),\dots,T(\mathbf e_n)\}.$$

Sus dimensiones tienen nombre: la **nulidad** $\nu(T)=\dim\operatorname{Nu}(T)$ y el **rango** $\rho(T)=\dim\operatorname{Im}(T)$.

> **Sobre las palabras.** Muchos textos llaman *rango* tanto al conjunto $\operatorname{Im}(T)$ como a su dimensión. En este cuaderno, $\operatorname{Im}(T)$ es el conjunto y "rango" es el número $\rho(T)$.

**Teorema de la dimensión (rango–nulidad).**
$$\nu(T)+\rho(T)=n\quad(\text{la dimensión del dominio}).$$
Las $n$ dimensiones de la entrada se reparten entre las que **se pierden** (núcleo) y las que **sobreviven** (imagen).

**Dos propiedades clave.**
1. $T(\mathbf u)=T(\mathbf v)\iff \mathbf u-\mathbf v\in\operatorname{Nu}(T)$, porque $T(\mathbf u)-T(\mathbf v)=T(\mathbf u-\mathbf v)$.
2. $T$ es **inyectiva** (no confunde entradas distintas) $\iff\operatorname{Nu}(T)=\{\mathbf 0\}$.

**Cómo se calculan (Gauss–Jordan).** Reduce $A$ a su forma escalonada reducida. Las columnas con pivote corresponden a variables básicas y las demás a variables **libres**.
* **Base del núcleo:** una solución de $A\mathbf x=\mathbf 0$ por cada variable libre (esa variable vale 1 y las otras libres valen 0).
* **Base de la imagen:** las columnas **de la matriz original $A$** que tienen pivote.
* $\rho$ = número de pivotes y $\nu$ = número de variables libres. ¡El teorema de la dimensión es contar columnas!

### Un ejemplo a mano

$$
A_{ej}=\begin{bmatrix}1&0&1\\0&1&0\\1&1&1\end{bmatrix}
\ \xrightarrow{\ F_3-F_1-F_2\ }\
\begin{bmatrix}1&0&1\\0&1&0\\0&0&0\end{bmatrix}.
$$

Hay pivotes en las columnas 1 y 2; la variable $x_3=t$ es libre. De $x_1+x_3=0$ y $x_2=0$ sale $\mathbf x=t\,(-1,0,1)$. Entonces

$$
\operatorname{Nu}(A_{ej})=\operatorname{gen}\{(-1,0,1)\},\qquad
\operatorname{Im}(A_{ej})=\operatorname{gen}\{(1,0,1),\,(0,1,1)\},
$$

con $\nu=1$, $\rho=2$ y $1+2=3$ ✓. La imagen es un plano: un vector $(a,b,c)$ está en él exactamente cuando $c=a+b$ (compruébalo con las dos columnas).

La siguiente celda programa este procedimiento. **Léela con calma**: es Gauss–Jordan, como lo haces a mano.

In [ ]:
def rref(A, tol=1e-9):
    """Forma escalonada reducida de A (Gauss–Jordan). Devuelve (R, columnas_pivote)."""
    R = np.array(A, dtype=float)
    m, n = R.shape
    tol = tol * max(1.0, np.abs(R).max())
    pivotes, fila = [], 0
    for col in range(n):
        if fila >= m:
            break
        k = fila + np.argmax(np.abs(R[fila:, col]))   # el mejor pivote de la columna (pivoteo parcial)
        if abs(R[k, col]) < tol:
            continue                                  # columna sin pivote → variable libre
        R[[fila, k]] = R[[k, fila]]                   # intercambiar filas
        R[fila] = R[fila] / R[fila, col]              # hacer el pivote igual a 1
        for i in range(m):
            if i != fila:
                R[i] = R[i] - R[i, col] * R[fila]     # ceros arriba y abajo del pivote
        pivotes.append(col)
        fila += 1
    R[np.abs(R) < tol] = 0.0
    return R, pivotes

def base_nucleo(A):
    """Base del núcleo: una solución de Ax = 0 por cada variable libre (columnas de la matriz devuelta)."""
    R, piv = rref(A)
    n = R.shape[1]
    libres = [j for j in range(n) if j not in piv]
    base = []
    for f in libres:
        x = np.zeros(n)
        x[f] = 1.0                                    # la variable libre vale 1
        for i, p in enumerate(piv):
            x[p] = -R[i, f]                           # las básicas se despejan
        base.append(x)
    return np.array(base).T if base else np.zeros((n, 0))

def base_imagen(A):
    """Base de la imagen: las columnas de la matriz ORIGINAL que tienen pivote."""
    _, piv = rref(A)
    return np.asarray(A, dtype=float)[:, piv]

def fmt(v):
    """Escribe un vector en forma compacta, por ejemplo (-2, 1, 0)."""
    t = []
    for x in np.ravel(v):
        s = f"{x:.3f}".rstrip("0").rstrip(".") if abs(x) > 1e-9 else "0"
        t.append("0" if s in ("-0", "") else s)
    return "(" + ", ".join(t) + ")"

def analizar(A, nombre="T", mostrar_matriz=True):
    """Imprime el rango, la nulidad y bases del núcleo y de la imagen de T(x) = A x."""
    A = np.asarray(A, dtype=float)
    m, n = A.shape
    N, B = base_nucleo(A), base_imagen(A)
    nu, rho = N.shape[1], B.shape[1]
    print(f"{nombre}: R^{n} → R^{m}")
    if mostrar_matriz:
        print("  matriz =\n " + str(A).replace("\n", "\n "))
    print(f"  rango   ρ = dim Im = {rho}")
    print(f"  nulidad ν = dim Nu = {nu}")
    print("  Base del núcleo:", ", ".join(fmt(N[:, j]) for j in range(nu)) if nu else "Nu = {0} (solo el vector cero)")
    if m <= 6:
        print("  Base de la imagen:", ", ".join(fmt(B[:, j]) for j in range(rho)) if rho else "Im = {0}")
    else:                                            # vectores muy largos: decimos cuáles columnas son
        piv = rref(A)[1]
        print(f"  Base de la imagen: las columnas {', '.join(str(p + 1) for p in piv)} de la matriz (vectores de R^{m})")
    print(f"  Teorema de la dimensión: ν + ρ = {nu} + {rho} = {nu + rho} = n ✓")

def comprobar_nucleo(A, v):
    """¿Está v en el núcleo de A?"""
    if v is None:
        print("✍️ Escribe tu vector en la variable (por ejemplo np.array([1, 0, 0])) y vuelve a ejecutar.")
        return
    v = np.asarray(v, float); Av = np.asarray(A, float) @ v
    if np.allclose(v, 0):
        print("El vector cero siempre está en el núcleo. Busca uno distinto de cero.")
    elif np.allclose(Av, 0, atol=1e-9):
        print(f"✓ Sí: A·{fmt(v)} = {fmt(Av)}. Tu vector está en el núcleo.")
    else:
        print(f"✗ Todavía no: A·{fmt(v)} = {fmt(Av)}, que no es el vector cero.")

def comprobar_imagen(A, w):
    """¿Está w en la imagen de A? (¿Es consistente el sistema A x = w?)"""
    if w is None:
        print("✍️ Escribe tu vector en la variable y vuelve a ejecutar.")
        return
    A = np.asarray(A, float); w = np.asarray(w, float)
    if len(rref(np.column_stack([A, w]))[1]) == len(rref(A)[1]):
        print(f"✓ Sí: {fmt(w)} está en la imagen (el sistema A x = w es consistente).")
    else:
        print(f"✗ No: {fmt(w)} no está en la imagen (A x = w es inconsistente: aparece un pivote en la última columna).")

print("Herramientas de núcleo e imagen listas.")

In [ ]:
A_ej = np.array([[1.0, 0.0, 1.0],
                 [0.0, 1.0, 0.0],
                 [1.0, 1.0, 1.0]])
R_ej, piv = rref(A_ej)
print("Forma escalonada reducida:\n", R_ej)
print("Columnas pivote (contando desde 0):", piv, "\n")
analizar(A_ej, "A_ej")

🔮 Con $\nu=1$ y $\rho=2$, ¿cómo crees que se verá la casa transformada por $A_{ej}$? ¿En qué dirección se "aplasta"?

La siguiente función muestra a la izquierda el **dominio**, con la casa y el núcleo (en rojo), y a la derecha el **codominio**, con la casa transformada y la imagen (en azul).

In [ ]:
def ver_nucleo_imagen(A, nombre="T", elev=22, azim=-58):
    """Izquierda: la casa y Nu(T). Derecha: T(casa) e Im(T)."""
    A = np.asarray(A, float); W = A @ V_CASA
    N, B = base_nucleo(A), base_imagen(A)
    extra = [s * 2.5 * N[:, j] / np.linalg.norm(N[:, j]) for j in range(N.shape[1]) for s in (1, -1)]
    lo1, hi1 = limites([V_CASA] + extra)            # el dominio se dibuja con espacio para ver el núcleo
    lo2, hi2 = limites([V_CASA, W])
    fig = plt.figure(figsize=(11.5, 5))
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    preparar_ejes(ax1, lo1, hi1, f"Dominio: la casa y Nu({nombre})", elev, azim)
    if N.shape[1] in (1, 2):
        dibujar_subespacio(ax1, N, lo1, hi1, COLOR_NUCLEO, alpha=0.15)
    dibujar_casa(ax1, V_CASA, alpha=0.55)
    ax1.legend(handles=[Line2D([], [], color=COLOR_NUCLEO, ls="--", lw=2, label=f"Nu({nombre})")], loc="upper left", fontsize=8)
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    lo, hi = lo2, hi2
    preparar_ejes(ax2, lo, hi, f"Codominio: {nombre}(casa) e Im({nombre})", elev, azim)
    if B.shape[1] in (1, 2):
        dibujar_subespacio(ax2, B, lo, hi, COLOR_IMAGEN, alpha=0.18)
    dibujar_casa(ax2, V_CASA, alpha=0.06, color="0.5", lw=0.3)
    dibujar_casa(ax2, W, alpha=0.9, zorder=2)
    ax2.legend(handles=[Patch(color=COLOR_IMAGEN, alpha=0.35, label=f"Im({nombre})")], loc="upper left", fontsize=8)
    plt.tight_layout(); plt.show()

ver_nucleo_imagen(A_ej, "A_ej")

Ahora comprobemos la propiedad clave $T(\mathbf u)=T(\mathbf v)\iff\mathbf u-\mathbf v\in\operatorname{Nu}(T)$: tomamos un vértice $\mathbf u$ de la casa, le sumamos un múltiplo del vector del núcleo y miramos las imágenes.

In [ ]:
k = base_nucleo(A_ej)[:, 0]            # el vector (-1, 0, 1) que genera el núcleo
u = V_CASA[:, 9]                       # un vértice: el extremo trasero de la cumbrera del techo
for t in [0.0, 1.5, -2.0]:
    v = u + t * k
    print(f"v = u + ({t:+.1f})k = {fmt(v):<22}  →  A_ej v = {fmt(A_ej @ v)}")

### 🤔 Preguntas para pensar (1.6)

a) Sin calcular: ¿cuál es el núcleo de la rotación $R_{60}$? ¿Y su imagen? Comprueba con `analizar(R60, "R60")`.

b) Explica por qué el núcleo siempre contiene a $\mathbf 0$ y por qué es un subespacio: si $\mathbf u,\mathbf v\in\operatorname{Nu}(T)$, ¿están también $\mathbf u+\mathbf v$ y $c\,\mathbf u$?

c) Verifica a mano que $(-1,0,1)\in\operatorname{Nu}(A_{ej})$ y que $(1,1,2)\in\operatorname{Im}(A_{ej})$. Encuentra un $\mathbf x$ con $A_{ej}\mathbf x=(1,1,2)$. ¿Es el único?

d) ¿Está $(1,0,0)$ en $\operatorname{Im}(A_{ej})$? Justifica de dos maneras distintas.

e) En la figura, la casa transformada quedó plana. ¿Qué nulidad produce una casa plana? ¿Y una casa reducida a un segmento? ¿Y una casa reducida a un punto?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) $\operatorname{Nu}(R_{60})=\{\mathbf 0\}$, porque el único vector que una rotación manda al origen es el propio origen (y la rotación se puede deshacer con $R_{-60}$). Por el teorema, $\rho=3-0=3$: $\operatorname{Im}(R_{60})=\mathbb{R}^3$.

b) $T(\mathbf 0)=\mathbf 0$, así que $\mathbf 0\in\operatorname{Nu}(T)$. Si $T(\mathbf u)=\mathbf 0$ y $T(\mathbf v)=\mathbf 0$, entonces $T(\mathbf u+\mathbf v)=T(\mathbf u)+T(\mathbf v)=\mathbf 0$ y $T(c\mathbf u)=c\,T(\mathbf u)=\mathbf 0$. El núcleo es cerrado bajo sumas y múltiplos: es un subespacio.

c) $A_{ej}(-1,0,1)=(-1+1,\ 0,\ -1+0+1)=(0,0,0)$ ✓. Además $A_{ej}(1,1,0)=(1,1,2)$, así que $(1,1,2)$ sí está en la imagen. No es la única solución: todos los vectores $(1,1,0)+t(-1,0,1)$ sirven (por ejemplo $(0,1,1)$), porque difieren en un vector del núcleo (propiedad 1).

d) No está. Primera manera: la imagen es el plano $c=a+b$, y para $(1,0,0)$ se tiene $0\neq 1+0$. Segunda manera: en $A_{ej}\mathbf x=(1,0,0)$ la tercera fila de $A_{ej}$ es la suma de las dos primeras, así que la tercera componente del resultado tendría que ser $1+0=1$, pero es $0$: el sistema es inconsistente. (Tercera manera: `comprobar_imagen(A_ej, [1, 0, 0])`.)

e) Para $T:\mathbb{R}^3\to\mathbb{R}^3$: con $\nu=1$ se tiene $\rho=2$ (casa plana); con $\nu=2$, $\rho=1$ (segmento); con $\nu=3$, $\rho=0$ (todo cae en el origen: $A$ es la matriz cero).

</details>

✍️ **Tu turno: comprueba tus respuestas.** Escribe un vector en lugar de `None` y ejecuta.

In [ ]:
mi_vector_nucleo = None          # por ejemplo: np.array([2, 0, -2])
comprobar_nucleo(A_ej, mi_vector_nucleo)

mi_vector_imagen = None          # por ejemplo: np.array([1, 0, 0])
comprobar_imagen(A_ej, mi_vector_imagen)

## 1.7 Proyectar la casa sobre un plano

Los arquitectos describen una casa con dibujos planos: la **planta** (vista desde arriba), el **alzado** o fachada (vista de frente) y el **perfil** (vista de lado). Cada uno es una **proyección**: una transformación lineal que aplasta la casa sobre un plano.

Las proyecciones son ideales para ver núcleo e imagen:

* la **imagen** es el plano donde cae la figura;
* el **núcleo** es la dirección en la que se aplasta.

### 1.7.1 La planta: proyección sobre el piso

$$
Q=\begin{bmatrix}1&0&0\\0&1&0\\0&0&0\end{bmatrix},\qquad Q(x,y,z)=(x,y,0).
$$

🔮 Antes de ejecutar: ¿cuáles son $\operatorname{Nu}(Q)$ e $\operatorname{Im}(Q)$? ¿Qué les pasará a las aristas verticales de la casa y a la chimenea?

In [ ]:
def ver_proyeccion(A, titulo="", ax=None, V=None, elev=22, azim=-58, original_solida=False, color=None, extra_nucleo=1.8):
    """Una sola figura 3D: la casa, su proyección A·V, los 'rayos' x → A x, la imagen (azul) y el núcleo (rojo)."""
    V = V_CASA if V is None else V
    A = np.asarray(A, float); W = A @ V
    N = base_nucleo(A)
    extra = [s * extra_nucleo * N[:, j] / np.linalg.norm(N[:, j]) for j in range(N.shape[1]) for s in (1, -1)]
    lo, hi = limites([V, W] + extra)
    if ax is None:
        ax = plt.figure(figsize=(6.5, 5.6)).add_subplot(projection="3d")
    preparar_ejes(ax, lo, hi, titulo, elev, azim)
    dibujar_subespacio(ax, base_imagen(A), lo, hi, COLOR_IMAGEN, alpha=0.15)
    dibujar_rayos(ax, V, W)
    dibujar_casa(ax, W, alpha=0.75 if color else 0.95, color=color, zorder=1.5)
    dibujar_casa(ax, V, alpha=0.85 if original_solida else 0.12, color=None if original_solida else "0.5",
                 lw=0.9 if original_solida else 0.3, zorder=2)
    if N.shape[1] in (1, 2):
        dibujar_subespacio(ax, N, lo, hi, COLOR_NUCLEO, alpha=0.12)
    leyenda_nucleo_imagen(ax)
    return ax

Q_planta = np.diag([1.0, 1.0, 0.0])
analizar(Q_planta, "Q (planta)")

fig = plt.figure(figsize=(12, 5))
ver_proyeccion(Q_planta, "La casa y su proyección sobre el piso", ax=fig.add_subplot(1, 2, 1, projection="3d"), extra_nucleo=0)
ax = fig.add_subplot(1, 2, 2)
W = Q_planta @ V_CASA
dibujar_2d(ax, W[[0, 1]], v=np.array([0.0, 0.0, 1.0]), titulo="La planta en el papel: coordenadas (x, y) de Q·V")
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.tight_layout(); plt.show()

### Dos casas distintas, una sola planta

La casa B es igual a la casa A pero tiene el techo más alto (3.4 en vez de 2.5). 🔮 ¿Tendrán la misma planta?

In [ ]:
V_ALTA, _, _ = construir_casa(altura_techo=3.4)
lo, hi = limites([V_CASA, V_ALTA])
fig = plt.figure(figsize=(14, 4.8))
for k, (Vk, t) in enumerate([(V_CASA, "Casa A"), (V_ALTA, "Casa B (techo más alto)")]):
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    preparar_ejes(ax, lo, hi, t)
    dibujar_rayos(ax, Vk, Q_planta @ Vk)
    dibujar_casa(ax, Q_planta @ Vk, color="0.45", alpha=0.5)
    dibujar_casa(ax, Vk, zorder=2)
ax = fig.add_subplot(1, 3, 3)
dibujar_2d(ax, (Q_planta @ V_CASA)[[0, 1]], v=np.array([0.0, 0.0, 1.0]), titulo="Planta de A (color) y de B (línea punteada)")
dibujar_2d(ax, (Q_planta @ V_ALTA)[[0, 1]], V3=V_ALTA, v=np.array([0.0, 0.0, 1.0]), contorno=True, ls="--")
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.tight_layout(); plt.show()

D = V_ALTA - V_CASA
print("¿Q·V_A = Q·V_B?                    ", np.allclose(Q_planta @ V_CASA, Q_planta @ V_ALTA))
print("Diferencias V_B − V_A en x y en y:  ", np.unique(np.round(D[:2], 9)))
print("¿Todas las columnas de V_B − V_A están en Nu(Q)?", np.allclose(Q_planta @ D, 0))

### 🤔 Preguntas para pensar (1.7.1)

a) ¿Qué aristas de la casa se convierten en un solo punto en la planta? Explica por qué, usando el núcleo.

b) Con solo la planta, ¿puedes reconstruir la casa? Relaciona tu respuesta con la inyectividad y con el ejemplo de las casas A y B.

c) Calcula $Q^2$ (✍️ `Q_planta @ Q_planta`). ¿Qué significa geométricamente "proyectar dos veces"?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) Las aristas verticales: las esquinas de las paredes y los lados de la chimenea. Sus extremos $\mathbf u,\mathbf v$ difieren en un múltiplo de $\mathbf e_3$, que está en $\operatorname{Nu}(Q)$; por la propiedad 1, $Q\mathbf u=Q\mathbf v$.

b) No. Como $\operatorname{Nu}(Q)\neq\{\mathbf 0\}$, $Q$ no es inyectiva: todas las casas que difieren solo en las alturas (como A y B, cuyos vértices difieren en vectores verticales) tienen la misma planta. La información sobre la coordenada $z$ se perdió: vive en el núcleo.

c) $Q^2=Q$. Un punto que ya está en el piso no se mueve si lo proyectas otra vez: $Q$ deja fija su imagen ($Q\mathbf y=\mathbf y$ para todo $\mathbf y\in\operatorname{Im}(Q)$). Esta propiedad, $P^2=P$, caracteriza a las proyecciones.

</details>

### 1.7.2 Las tres vistas del arquitecto

$$
Q_{\text{planta}}=\begin{bmatrix}1&0&0\\0&1&0\\0&0&0\end{bmatrix},\quad
Q_{\text{alzado}}=\begin{bmatrix}1&0&0\\0&0&0\\0&0&1\end{bmatrix},\quad
Q_{\text{perfil}}=\begin{bmatrix}0&0&0\\0&1&0\\0&0&1\end{bmatrix}.
$$

Para dibujarlas en el papel usamos las dos coordenadas que no se anulan.

In [ ]:
Q_alzado = np.diag([1.0, 0.0, 1.0])
Q_perfil = np.diag([0.0, 1.0, 1.0])
vistas = [("Planta (desde arriba)", Q_planta, [0, 1], np.array([0.0, 0.0, 1.0]), ("x", "y")),
          ("Alzado (desde el frente)", Q_alzado, [0, 2], np.array([0.0, -1.0, 0.0]), ("x", "z")),
          ("Perfil (desde la derecha)", Q_perfil, [1, 2], np.array([1.0, 0.0, 0.0]), ("y", "z"))]
fig, axs = plt.subplots(1, 3, figsize=(13, 4.3))
for ax, (t, Qv, filas, v, eti) in zip(axs, vistas):
    dibujar_2d(ax, (Qv @ V_CASA)[filas], v=v, titulo=t)
    ax.set_xlabel(eti[0]); ax.set_ylabel(eti[1])
plt.tight_layout(); plt.show()
for t, Qv, *_ in vistas:
    print(f"{t:<27} núcleo generado por {fmt(base_nucleo(Qv)[:, 0])}")

### 🤔 Preguntas para pensar (1.7.2)

a) ¿Qué dirección se pierde en cada vista? Relaciónalo con el núcleo que imprimió la celda.

b) Cada vista por separado pierde información. ¿Por qué las tres juntas sí permiten reconstruir cada vértice? Pista: ¿qué vectores están en los tres núcleos a la vez?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) La planta pierde la altura ($\operatorname{Nu}=$ eje $z$), el alzado pierde la profundidad ($\operatorname{Nu}=$ eje $y$) y el perfil pierde el ancho ($\operatorname{Nu}=$ eje $x$).

b) El único vector que está en los tres núcleos es $\mathbf 0$. Si juntamos las tres vistas en una sola transformación $\mathbf x\mapsto(Q_{\text{planta}}\mathbf x,\ Q_{\text{alzado}}\mathbf x,\ Q_{\text{perfil}}\mathbf x)$, su núcleo es la intersección de los tres núcleos, es decir $\{\mathbf 0\}$: esa transformación sí es inyectiva. En la práctica, cada coordenada aparece en dos de las tres vistas. (Hace falta saber qué punto de una vista corresponde a qué punto de otra; por eso los planos de arquitectura se dibujan alineados.)

</details>

### 1.7.3 La sombra de la tarde: una proyección oblicua

El sol ilumina la casa con rayos paralelos en la dirección $\mathbf d=(d_1,d_2,d_3)$, con $d_3<0$ (los rayos bajan). Cada punto $\mathbf x$ viaja por su rayo, $\mathbf x+t\,\mathbf d$, hasta llegar al piso ($z=0$), lo que ocurre cuando $t=-x_3/d_3$:

$$
S(\mathbf x)=\mathbf x-\frac{x_3}{d_3}\,\mathbf d,
\qquad
S=\begin{bmatrix}1&0&-d_1/d_3\\0&1&-d_2/d_3\\0&0&0\end{bmatrix}.
$$

Es lineal, y su imagen también es el piso $z=0$, igual que la planta.

🔮 La sombra y la planta caen en el mismo plano. ¿Son la misma figura? ¿Tienen el mismo núcleo?

In [ ]:
d_sol = np.array([1.0, 0.6, -1.2])             # ✍️ cambia la dirección del sol (con d3 < 0)

S_sol = np.array([[1.0, 0.0, -d_sol[0] / d_sol[2]],
                  [0.0, 1.0, -d_sol[1] / d_sol[2]],
                  [0.0, 0.0, 0.0]])
print("¿La función sombra es lineal?")
es_lineal(lambda x: x - x[2] / d_sol[2] * d_sol)
print("S d =", fmt(S_sol @ d_sol), "  ← ¡la dirección del sol está en el núcleo!\n")
analizar(S_sol, "S (sombra)")

fig = plt.figure(figsize=(12.5, 5.2))
ver_proyeccion(S_sol, "La casa al sol y su sombra en el piso", ax=fig.add_subplot(1, 2, 1, projection="3d"),
               original_solida=True, color="0.3")
ax = fig.add_subplot(1, 2, 2)
dibujar_2d(ax, (S_sol @ V_CASA)[[0, 1]], color="0.55", bordes=False, titulo="Sombra (gris) y planta (línea punteada)")
dibujar_2d(ax, (Q_planta @ V_CASA)[[0, 1]], contorno=True, ls="--")
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.tight_layout(); plt.show()

### 🤔 Preguntas para pensar (1.7.3)

a) La planta y la sombra tienen la **misma imagen** (el piso) y **distinto núcleo**. ¿Qué decide entonces la forma de la figura proyectada: el plano donde cae, la dirección en que se aplasta o ambos?

b) Verifica a mano que $S\mathbf d=\mathbf 0$. ¿Qué significa esto físicamente?

c) ¿Qué sombra produce el sol del mediodía, $\mathbf d=(0,0,-1)$? Escribe la matriz.

d) Al atardecer, $d_3$ se acerca a $0$ (los rayos son casi horizontales). ¿Qué pasa con la tercera columna de $S$ y con la sombra? ¿Qué pasa si $d_3=0$?

e) ✍️ Cambia `d_sol` en la celda anterior. ¿Qué dirección del sol hace que la sombra quede detrás de la casa (hacia $y$ positivo)?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) Ambos. La imagen dice **dónde** cae la figura y el núcleo dice **en qué dirección** se aplasta. La planta y la sombra caen en el mismo plano, pero la sombra se estira en la dirección de $\mathbf d$, que es su núcleo. Una proyección queda determinada por su imagen y su núcleo.

b) $S\mathbf d=\mathbf d-\dfrac{d_3}{d_3}\mathbf d=\mathbf 0$. Físicamente: todos los puntos de un mismo rayo de luz caen en el mismo punto de sombra, porque su diferencia es un múltiplo de $\mathbf d$, que está en el núcleo.

c) Con $\mathbf d=(0,0,-1)$ las entradas $-d_1/d_3$ y $-d_2/d_3$ son cero: $S=\begin{bmatrix}1&0&0\\0&1&0\\0&0&0\end{bmatrix}=Q_{\text{planta}}$. Al mediodía, la sombra es la planta.

d) Las entradas $-d_1/d_3$ y $-d_2/d_3$ crecen sin límite: la sombra se alarga muchísimo. Si $d_3=0$, los rayos son horizontales y nunca llegan al piso: la fórmula divide por cero y no hay tal proyección. (La dirección $\mathbf d$ quedaría dentro del propio piso, y una proyección no puede tener un vector no nulo que esté a la vez en su núcleo y en su imagen.)

e) Basta que el sol "empuje" hacia $y$ positivo: por ejemplo $\mathbf d=(0,\,1,\,-1)$. El desplazamiento de cada punto es $-\dfrac{x_3}{d_3}(d_1,d_2)$, que con $d_3<0$ tiene el sentido de $(d_1,d_2)$.

</details>

### 1.7.4 Proyección ortogonal sobre un plano inclinado

Para proyectar perpendicularmente sobre el plano $\mathbf n\cdot\mathbf x=0$, a cada vector le quitamos su componente en la dirección de $\mathbf n$ (la proyección de $\mathbf x$ sobre $\mathbf n$, que ya conoces del producto punto):

$$
P\mathbf x=\mathbf x-\frac{\mathbf x\cdot\mathbf n}{\mathbf n\cdot\mathbf n}\,\mathbf n,
\qquad
P=I-\frac{\mathbf n\,\mathbf n^{T}}{\mathbf n^{T}\mathbf n}.
$$

Usaremos $\mathbf n=(1,2,2)$, es decir, el plano $x+2y+2z=0$.

In [ ]:
n_plano = np.array([1.0, 2.0, 2.0])
P_plano = np.eye(3) - np.outer(n_plano, n_plano) / (n_plano @ n_plano)
analizar(P_plano, "P (ortogonal)")
print("\n¿P es simétrica (Pᵀ = P)?", np.allclose(P_plano.T, P_plano))
print("¿P² = P?                 ", np.allclose(P_plano @ P_plano, P_plano))
ver_proyeccion(P_plano, "Proyección ortogonal sobre el plano x + 2y + 2z = 0", elev=28, azim=15)
plt.show()

### 🤔 Preguntas para pensar (1.7.4)

a) Usa la fórmula $P\mathbf x=\mathbf x-\frac{\mathbf x\cdot\mathbf n}{\mathbf n\cdot\mathbf n}\mathbf n$ para explicar por qué $\operatorname{Nu}(P)$ es la recta generada por $\mathbf n$ y por qué $\operatorname{Im}(P)$ es el plano $\mathbf n\cdot\mathbf x=0$. ¿El vector del núcleo que imprimió la celda es paralelo a $\mathbf n$?

b) Compara los rayos punteados de esta figura con los de la sombra. ¿Qué ángulo forman con el plano en cada caso? ¿Por qué esta se llama proyección **ortogonal**?

c) Calcula a mano $P\mathbf w$ para $\mathbf w=(2,-1,0)$. ¿Por qué el resultado era de esperarse?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) $P\mathbf x=\mathbf 0$ si y solo si $\mathbf x=\frac{\mathbf x\cdot\mathbf n}{\mathbf n\cdot\mathbf n}\mathbf n$, es decir, si y solo si $\mathbf x$ es múltiplo de $\mathbf n$: el núcleo es la recta generada por $\mathbf n$. Para la imagen: $\mathbf n\cdot(P\mathbf x)=\mathbf n\cdot\mathbf x-\frac{\mathbf x\cdot\mathbf n}{\mathbf n\cdot\mathbf n}(\mathbf n\cdot\mathbf n)=0$, así que toda imagen está en el plano; y si $\mathbf w$ está en el plano, $P\mathbf w=\mathbf w$, así que todo el plano es imagen. La celda imprimió $(0.5,1,1)=\tfrac12(1,2,2)$, paralelo a $\mathbf n$ ✓.

b) Aquí los rayos son paralelos a $\mathbf n$: forman $90^\circ$ con el plano. En la sombra los rayos son paralelos a $\mathbf d$, que no es perpendicular al piso: es una proyección **oblicua**. "Ortogonal" significa que el núcleo es perpendicular a la imagen.

c) $\mathbf w\cdot\mathbf n=2-2+0=0$, así que $\mathbf w$ ya está en el plano y $P\mathbf w=\mathbf w=(2,-1,0)$. Una proyección deja fijos los vectores de su imagen.

</details>

### 1.7.5 De 3D a 2D: la "foto" de la casa

Una foto (sin perspectiva) es una transformación lineal $F:\mathbb{R}^3\to\mathbb{R}^2$. Si la cámara mira en la dirección del vector unitario $\mathbf v$ (que apunta hacia el observador), escogemos dos vectores unitarios perpendiculares a $\mathbf v$ y entre sí: $\mathbf u$ ("derecha" en la foto) y $\mathbf w$ ("arriba"). Entonces

$$
F=\begin{bmatrix}\ \mathbf u^{T}\ \\ \ \mathbf w^{T}\ \end{bmatrix}\ (2\times 3),\qquad F\mathbf x=(\mathbf u\cdot\mathbf x,\ \mathbf w\cdot\mathbf x).
$$

> Así dibuja `matplotlib` las figuras 3D de este cuaderno (configuramos la proyección "ortográfica"): una transformación lineal de $\mathbb{R}^3$ en $\mathbb{R}^2$, más un cambio de escala y una traslación para ubicar la imagen en la pantalla. Las cámaras reales añaden **perspectiva** (lo lejano se ve más pequeño), que ya no es lineal.

🔮 $F$ va de $\mathbb{R}^3$ a $\mathbb{R}^2$. ¿Cuál será su rango? ¿Y su nulidad?

In [ ]:
def matriz_foto(azim, elev):
    """Matriz 2x3 de una foto tomada desde la dirección (azim, elev), en grados, y el vector v hacia la cámara."""
    az, el = np.radians(azim), np.radians(elev)
    v = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])   # hacia la cámara
    u = np.array([-np.sin(az), np.cos(az), 0.0])                                  # "derecha" en la foto
    w = np.cross(v, u)                                                             # "arriba" en la foto
    return np.vstack([u, w]), v

fig, axs = plt.subplots(1, 3, figsize=(13, 4.3))
for ax, (az, el) in zip(axs, [(-58, 22), (30, 35), (-125, 12)]):
    F, v = matriz_foto(az, el)
    dibujar_2d(ax, F @ V_CASA, v=v, titulo=f"Foto F·V desde azim = {az}°, elev = {el}°")
plt.tight_layout(); plt.show()

F, v_camara = matriz_foto(-58, 22)
analizar(F, "F (foto)")
print("\nDirección de la cámara v =", fmt(v_camara), "  y  F v =", fmt(F @ v_camara))

### 🤔 Preguntas para pensar (1.7.5)

a) ¿Cuál es el codominio de $F$ y cuál es su imagen? ¿Coinciden? ¿Es $F$ sobreyectiva?

b) Aplica el teorema de la dimensión a $F$. ¿Por qué el teorema habla de la dimensión del **dominio** y no de la del codominio?

c) ¿Puede una transformación lineal de $\mathbb{R}^3$ en $\mathbb{R}^2$ ser inyectiva? Justifica con el teorema.

d) El núcleo de $F$ es la recta generada por $\mathbf v$. Interpreta: ¿qué puntos de la escena "se tapan" entre sí en la foto?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) El codominio es $\mathbb{R}^2$. Como $\rho(F)=2$, la imagen es todo $\mathbb{R}^2$: coinciden y $F$ es sobreyectiva. (En cambio, en la planta $Q$, el codominio es $\mathbb{R}^3$ y la imagen es solo un plano.)

b) $\nu+\rho=1+2=3=\dim(\text{dominio})$. El teorema reparte las dimensiones de la **entrada**: algunas se pierden (núcleo) y otras llegan a la salida (imagen). El codominio puede ser más grande que la imagen, así que su dimensión no entra en la cuenta.

c) No. $\rho\le 2$ porque la imagen está dentro de $\mathbb{R}^2$; entonces $\nu=3-\rho\ge 1$, el núcleo no es $\{\mathbf 0\}$ y $F$ no es inyectiva. Siempre se pierde al menos una dimensión: la profundidad.

d) Dos puntos $\mathbf x,\mathbf y$ caen en el mismo lugar de la foto exactamente cuando $\mathbf x-\mathbf y$ es múltiplo de $\mathbf v$, es decir, cuando están sobre la misma línea de visión de la cámara. Por eso un objeto puede tapar a otro.

</details>

### 1.7.6 Aplastar la casa sobre una recta

Proyectar ortogonalmente sobre la recta generada por $\mathbf u$:

$$
L\mathbf x=\frac{\mathbf x\cdot\mathbf u}{\mathbf u\cdot\mathbf u}\,\mathbf u,\qquad L=\frac{\mathbf u\,\mathbf u^{T}}{\mathbf u^{T}\mathbf u},\qquad \mathbf u=(2,1,1).
$$

🔮 ¿Qué forma tendrá la imagen de la casa? ¿Qué dimensión tendrá el núcleo?

In [ ]:
u_recta = np.array([2.0, 1.0, 1.0])
L_recta = np.outer(u_recta, u_recta) / (u_recta @ u_recta)
analizar(L_recta, "L (recta)")
ver_nucleo_imagen(L_recta, "L")

### 🤔 Preguntas para pensar (1.7.6)

a) ¿Por qué el núcleo de $L$ es un plano? Escribe su ecuación.

b) ¿Qué tienen en común todos los vértices de la casa que caen en el mismo punto del segmento?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) $L\mathbf x=\mathbf 0\iff \mathbf x\cdot\mathbf u=0$: el núcleo es el plano perpendicular a $\mathbf u$, de ecuación $2x+y+z=0$. Tiene dimensión 2, y $\nu+\rho=2+1=3$ ✓.

b) Tienen el mismo producto punto con $\mathbf u$ (la misma "sombra" sobre la recta). Su diferencia es perpendicular a $\mathbf u$, es decir, está en el núcleo.

</details>

## 1.8 Resumen de la Actividad 1

La siguiente tabla reúne todas las transformaciones que hemos usado.

In [ ]:
transformaciones = [("Escalamiento S", S), ("Rotación R60", R60), ("Reflexión M", M), ("Cizalla C", C),
                    ("Ejemplo A_ej", A_ej), ("Planta Q", Q_planta), ("Sombra S_sol", S_sol),
                    ("Ortogonal P", P_plano), ("Recta L", L_recta), ("Foto F (2×3)", F)]
print(f"{'Transformación':<17}{'R^n → R^m':>11}{'ρ':>4}{'ν':>4}    ¿inyectiva?  ¿sobreyectiva?  ν + ρ = n")
print("-" * 80)
for nombre, A_t in transformaciones:
    m, n = A_t.shape
    rho, nu = base_imagen(A_t).shape[1], base_nucleo(A_t).shape[1]
    iny = "sí" if nu == 0 else "no"
    sob = "sí" if rho == m else "no"
    print(f"{nombre:<17}{f'R^{n} → R^{m}':>11}{rho:>4}{nu:>4}    {iny:^11}  {sob:^14}  {nu} + {rho} = {n}")

### 🤔 Preguntas para pensar (1.8)

a) En la tabla, ¿hay alguna transformación con $\rho=3$ que aplaste la casa? ¿Por qué no puede haberla?

b) ¿Cuáles transformaciones de la tabla se pueden deshacer? ¿Qué tienen en común sus núcleos? Para $T:\mathbb{R}^3\to\mathbb{R}^3$, ¿por qué "inyectiva" y "sobreyectiva" aparecen siempre juntas?

c) Si primero rotas la casa y luego tomas la planta, ¿cuál es el rango de la composición? ¿Puede una composición tener mayor rango que alguna de las transformaciones que la forman? (✍️ Comprueba con `analizar(Q_planta @ R60)`.)

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) No. "Aplastar" significa que la imagen de la casa queda dentro de un plano o de una recta. Como los vértices de la casa generan $\mathbb{R}^3$ (la casa tiene volumen), sus imágenes generan toda $\operatorname{Im}(T)$; si $\rho=3$, las imágenes generan $\mathbb{R}^3$ y no caben en un plano.

b) $S$, $R_{60}$, $M$ y $C$: todas tienen $\operatorname{Nu}=\{\mathbf 0\}$. Para $T:\mathbb{R}^3\to\mathbb{R}^3$, el teorema dice $\nu+\rho=3$, así que $\nu=0\iff\rho=3$: inyectiva $\iff$ sobreyectiva. Las proyecciones no se pueden deshacer, porque lo que mandan al núcleo se pierde.

c) $\rho(Q_{\text{planta}}R_{60})=2$. Una composición nunca tiene mayor rango que sus partes: $\operatorname{Im}(BA)\subseteq\operatorname{Im}(B)$, así que $\rho(BA)\le\rho(B)$; y $\operatorname{Nu}(A)\subseteq\operatorname{Nu}(BA)$, así que $\rho(BA)\le\rho(A)$. Una vez aplastada la casa, ninguna transformación lineal posterior la "desaplasta".

</details>

### 🏁 Reto

Diseña una matriz $A$ de $3\times 3$ tal que:
* $\operatorname{Nu}(A)$ sea la recta generada por $(1,1,0)$, y
* $\operatorname{Im}(A)$ sea el plano $z=0$.

Pistas: ¿qué tienen que cumplir las columnas $A\mathbf e_1, A\mathbf e_2, A\mathbf e_3$ para que todas queden en el plano $z=0$? Si $A(1,1,0)=\mathbf 0$, ¿qué relación hay entre $A\mathbf e_1$ y $A\mathbf e_2$? ¿Es tu matriz una proyección (es decir, $A^2=A$)?

In [ ]:
def verificar_reto(A):
    if A is None:
        print("✍️ Escribe tu matriz en A_reto y vuelve a ejecutar.")
        return
    A = np.asarray(A, float)
    N, B = base_nucleo(A), base_imagen(A)
    ok_nucleo = N.shape[1] == 1 and np.allclose(A @ np.array([1.0, 1.0, 0.0]), 0)
    ok_imagen = B.shape[1] == 2 and np.allclose(A[2], 0)
    print("Núcleo = gen{(1,1,0)}:", "✓" if ok_nucleo else f"✗ (tu núcleo: {', '.join(fmt(N[:, j]) for j in range(N.shape[1])) or '{0}'})")
    print("Imagen = plano z = 0: ", "✓" if ok_imagen else f"✗ (tu rango es {B.shape[1]}; ¿la tercera fila es cero?)")
    if ok_nucleo and ok_imagen:
        print("¡Lo lograste! ¿Es una proyección? A² = A:", np.allclose(A @ A, A))
        ver_nucleo_imagen(A, "A_reto")

A_reto = None        # por ejemplo: np.array([[ , , ], [ , , ], [ , , ]])
verificar_reto(A_reto)

<details>
<summary><b>▶ Ver una solución posible</b> (inténtalo antes de abrir)</summary>

Una posibilidad: $A(x,y,z)=(x-y,\ z,\ 0)$, es decir,
$$A=\begin{bmatrix}1&-1&0\\0&0&1\\0&0&0\end{bmatrix}.$$
La tercera fila es cero, así que toda imagen está en $z=0$. Las columnas son $(1,0,0)$, $(-1,0,0)$ y $(0,1,0)$, que generan el plano $z=0$ ($\rho=2$). El núcleo es $\{x=y,\ z=0\}=\operatorname{gen}\{(1,1,0)\}$. No es una proyección: $A^2\neq A$. No toda transformación con un núcleo de dimensión 1 es una proyección.

</details>

---
# Actividad 2. Transformaciones lineales con datos

Las transformaciones lineales no solo mueven casas: también transforman **datos**. Cada vez que aplicas la misma fórmula lineal a cada píxel de una imagen o a cada fila de una tabla, estás calculando $T(\mathbf x)=A\mathbf x$. Las ideas de la Actividad 1 siguen funcionando:

* el **núcleo** dice qué información se **pierde** o qué es **redundante**;
* la **imagen** dice qué resultados son **posibles**.

## 2.1 Colores que se vuelven grises

En una imagen digital cada píxel es un color $\mathbf c=(r,g,b)$, con valores entre 0 (nada) y 1 (máximo) de rojo, verde y azul: **un vector de $\mathbb{R}^3$**. Para pasar una foto a blanco y negro, muchos programas (por ejemplo, la librería Pillow de Python) usan la **luminancia** de la norma BT.601:

$$
T(r,g,b)=0.299\,r+0.587\,g+0.114\,b=\begin{bmatrix}0.299&0.587&0.114\end{bmatrix}\begin{bmatrix}r\\g\\b\end{bmatrix}.
$$

Es una transformación lineal $T:\mathbb{R}^3\to\mathbb{R}$ cuya matriz es $1\times 3$. Los pesos reflejan que el ojo humano es más sensible al verde que al rojo, y mucho menos sensible al azul.

In [ ]:
a_gris = np.array([[0.299, 0.587, 0.114]])          # matriz 1 x 3

def muestras(colores, nombres, a=a_gris, titulo=""):
    """Arriba: cada color. Abajo: el gris que le asigna T(c) = a·c."""
    fig, ax = plt.subplots(figsize=(1.7 * len(colores) + 1, 2.8))
    for k, c in enumerate(colores):
        y = float(np.ravel(a) @ c)
        ax.add_patch(plt.Rectangle((k, 1.05), 0.9, 0.9, color=np.clip(c, 0, 1)))
        ax.add_patch(plt.Rectangle((k, 0.0), 0.9, 0.9, color=(np.clip(y, 0, 1),) * 3, ec="0.7"))
        ax.text(k + 0.45, -0.12, f"{nombres[k]}\nT(c) = {y:.3f}", ha="center", va="top", fontsize=9)
    ax.text(-0.1, 1.5, "color c", ha="right", va="center"); ax.text(-0.1, 0.45, "gris T(c)", ha="right", va="center")
    ax.set_xlim(-1.1, len(colores)); ax.set_ylim(-0.75, 2.0); ax.axis("off"); ax.set_title(titulo)
    plt.show()

colores = [np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0]), np.array([0.0, 0.0, 1.0]),
           np.array([1.0, 1.0, 0.0]), np.array([0.5, 0.5, 0.5])]
muestras(colores, ["rojo (1,0,0)", "verde (0,1,0)", "azul (0,0,1)", "amarillo (1,1,0)", "gris (.5,.5,.5)"],
         titulo="Cada color y su tono de gris")

🔮 $T$ va de $\mathbb{R}^3$ a $\mathbb{R}$. ¿Qué dimensión tendrá su núcleo? ¿Qué forma geométrica tiene?

In [ ]:
analizar(a_gris, "T (gris)")
K_gris = base_nucleo(a_gris)                         # columnas: una base del núcleo
print("\nComprobación: T(k1) =", (a_gris @ K_gris[:, 0]).item(), " y T(k2) =", (a_gris @ K_gris[:, 1]).item())

El núcleo es un **plano** de $\mathbb{R}^3$: son las direcciones en las que puedes cambiar un color **sin cambiar su gris**. Por la propiedad clave de la sección 1.6:

$$
\text{dos colores tienen el mismo gris}\iff T(\mathbf c_1)=T(\mathbf c_2)\iff \mathbf c_1-\mathbf c_2\in\operatorname{Nu}(T).
$$

Usemos esto para **esconder un número**. Partimos de un verde, $\mathbf c_{\text{fondo}}$, y le sumamos un vector del núcleo para obtener un rojizo, $\mathbf c_{\text{figura}}=\mathbf c_{\text{fondo}}+\mathbf k$, con $\mathbf k\in\operatorname{Nu}(T)$.

In [ ]:
c_fondo = np.array([0.42, 0.64, 0.32])               # un verde
k = -0.22 * K_gris[:, 0]                             # un vector del núcleo: más rojo y menos verde
c_figura = c_fondo + k
print("k        =", fmt(k), "  T(k) =", round((a_gris @ k).item(), 12) + 0.0)
print("c_fondo  =", c_fondo, "  T(c_fondo)  =", round((a_gris @ c_fondo).item(), 4))
print("c_figura =", c_figura, "  T(c_figura) =", round((a_gris @ c_figura).item(), 4))
muestras([c_fondo, c_figura], ["c_fondo", "c_figura"], titulo="Dos colores distintos con el mismo gris")

Ahora construimos una lámina de puntos, como las que se usan en los exámenes de visión de colores: los puntos dentro del número usan $\mathbf c_{\text{figura}}$ y los de afuera, $\mathbf c_{\text{fondo}}$. A cada punto le agregamos dos variaciones al azar: un poco de brillo (igual en los tres canales) y un poco de tono **en las direcciones del núcleo**.

🔮 ¿Se verá el número al pasar la imagen a gris? ¿Y si nos quedamos solo con el canal rojo, $T_{\text{rojo}}(r,g,b)=r$?

In [ ]:
# --- Herramientas para fabricar la lámina (no necesitas leerlas en detalle) ---
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

def mascara_texto(texto, tam=400):
    """Matriz booleana tam x tam: True donde está dibujado el texto (se dibuja fuera de pantalla)."""
    fig = Figure(figsize=(4, 4), dpi=tam / 4)
    FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.text(0.5, 0.46, texto, ha="center", va="center", fontsize=200, fontweight="bold")
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., 0] < 128

def lamina_puntos(color_figura, color_fondo, texto="7", tam=400, n_puntos=1500, semilla=3, variacion=None):
    """Imagen tam x tam x 3 hecha de puntos de colores dentro de un círculo."""
    rng = np.random.default_rng(semilla)
    mascara = mascara_texto(texto, tam)
    img = np.ones((tam, tam, 3))                                  # fondo blanco
    yy, xx = np.mgrid[0:tam, 0:tam]
    radio = tam / 2 - 4
    for _ in range(n_puntos):
        cx, cy = rng.uniform(0, tam, 2)
        if (cx - tam / 2) ** 2 + (cy - tam / 2) ** 2 > radio ** 2:
            continue
        r = rng.uniform(4.4, 10.6)
        color = (color_figura if mascara[int(cy), int(cx)] else color_fondo) + rng.uniform(-0.04, 0.04)
        if variacion is not None:                                 # variación de tono en las direcciones dadas
            color = color + variacion @ rng.uniform(-0.05, 0.05, variacion.shape[1])
        x0, x1 = int(max(cx - r, 0)), int(min(cx + r + 1, tam))
        y0, y1 = int(max(cy - r, 0)), int(min(cy + r + 1, tam))
        disco = (xx[y0:y1, x0:x1] - cx) ** 2 + (yy[y0:y1, x0:x1] - cy) ** 2 <= r ** 2
        img[y0:y1, x0:x1][disco] = color
    return np.clip(img, 0, 1)

K_unit = K_gris / np.linalg.norm(K_gris, axis=0)                  # direcciones del núcleo, de longitud 1
img = lamina_puntos(c_figura, c_fondo, variacion=K_unit)
print("La imagen es un arreglo de", img.shape, "→", img.shape[0] * img.shape[1], "píxeles, cada uno un vector de R^3")

In [ ]:
# Toda la imagen se transforma con un solo producto: cada píxel (r, g, b) ↦ a · (r, g, b)
gris = img @ a_gris.ravel()
rojo = img @ np.array([1.0, 0.0, 0.0])

fig, axs = plt.subplots(1, 3, figsize=(13, 4.6))
axs[0].imshow(img);                              axs[0].set_title("Imagen en color")
axs[1].imshow(gris, cmap="gray", vmin=0, vmax=1); axs[1].set_title("Gris: 0.299 r + 0.587 g + 0.114 b")
axs[2].imshow(rojo, cmap="gray", vmin=0, vmax=1); axs[2].set_title("Solo el canal rojo: T(r, g, b) = r")
for ax in axs:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 🤔 Preguntas para pensar (2.1)

a) Comprueba el teorema de la dimensión para $T(r,g,b)=0.299r+0.587g+0.114b$. ¿Cuál es su imagen?

b) Explica por qué el número desaparece en la imagen gris. Usa la frase "$\mathbf c_{\text{figura}}-\mathbf c_{\text{fondo}}$ está en…".

c) Los puntos de la lámina tienen tonos distintos (no todos los rojizos son iguales), pero en gris solo se nota una pequeña variación de brillo. ¿En qué direcciones variamos el color de los puntos? ¿Cuál de las dos variaciones sobrevive en gris?

d) La transformación "solo el canal rojo", $T_{\text{rojo}}(r,g,b)=r$, sí revela el número. ¿Cuál es su núcleo? ¿Está $\mathbf c_{\text{figura}}-\mathbf c_{\text{fondo}}$ en él?

e) Una forma simplificada de modelar la visión de una persona con daltonismo rojo-verde es como una transformación lineal que "no ve" ciertas diferencias de color. Con lo que aprendiste, explica cómo funcionan las láminas de puntos que se usan para detectarlo.

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) $\nu=2$ (el núcleo es un plano) y $\rho=1$: $2+1=3$ ✓. La imagen es todo $\mathbb{R}$ (para colores reales, con $r,g,b\in[0,1]$, el gris queda entre 0 y 1).

b) $\mathbf c_{\text{figura}}-\mathbf c_{\text{fondo}}=\mathbf k$ está en $\operatorname{Nu}(T)$, así que $T(\mathbf c_{\text{figura}})=T(\mathbf c_{\text{fondo}})$: el número y el fondo tienen exactamente el mismo gris. La diferencia entre ellos vivía en el núcleo y la transformación la borró.

c) Hicimos dos variaciones: una de brillo, en la dirección $(1,1,1)$, y otra de tono, en las direcciones del núcleo (`K_unit`). La de tono desaparece en gris porque está en el núcleo. La de brillo sobrevive, porque $T(1,1,1)=0.299+0.587+0.114=1\neq 0$; pero afecta por igual al número y al fondo, así que no delata la figura.

d) $\operatorname{Nu}(T_{\text{rojo}})=\{(0,g,b)\}$, el plano $r=0$. El vector $\mathbf c_{\text{figura}}-\mathbf c_{\text{fondo}}$ tiene componente roja $\approx 0.43\neq 0$, así que **no** está en ese núcleo: la diferencia sobrevive y el número se ve. El núcleo depende de la transformación: lo que una no ve, otra sí puede verlo.

e) Las láminas usan colores cuya diferencia está (aproximadamente) en el "núcleo" de esa visión: para quien ve normalmente, la diferencia sobrevive y el número aparece; para quien tiene esa forma de daltonismo, la diferencia cae en el núcleo de su transformación y el número se confunde con el fondo. (Es un modelo simplificado; la visión real es más compleja.)

</details>

✍️ **Tu turno.**

1. Cambia los pesos por el **promedio**, $T(r,g,b)=\tfrac13(r+g+b)$, que es otra forma común de pasar a gris. ¿Aparece el número? Usa `comprobar_nucleo` para explicar por qué.
2. Diseña tu propio par de colores con el mismo gris: elige `mi_fondo` y un vector de `K_gris` para sumarle (cuida que las componentes queden entre 0 y 1).

In [ ]:
a_prom = np.array([[1/3, 1/3, 1/3]])
plt.figure(figsize=(4.2, 4.2))
g_prom = img @ a_prom.ravel()
puntos = img.min(axis=2) < 0.99                       # píxeles de los puntos (no el fondo blanco)
plt.imshow(g_prom, cmap="gray", vmin=g_prom[puntos].min(), vmax=g_prom[puntos].max()); plt.axis("off")
plt.title("Gris como promedio (r + g + b)/3\n(con el contraste aumentado)"); plt.show()
print("¿La diferencia de colores está en el núcleo del promedio?")
comprobar_nucleo(a_prom, c_figura - c_fondo)

# Tu propio par de colores (cambia None por un color, por ejemplo np.array([0.3, 0.5, 0.7]))
mi_fondo = None
if mi_fondo is not None:
    mi_figura = mi_fondo + 0.15 * K_gris[:, 1]         # ✍️ prueba otras combinaciones de K_gris[:, 0] y K_gris[:, 1]
    muestras([mi_fondo, mi_figura], ["mi_fondo", "mi_figura"], titulo="Mi par de colores")
    plt.imshow(lamina_puntos(mi_figura, mi_fondo, texto="5")); plt.axis("off"); plt.show()

## 2.2 Una tabla de datos de viviendas

Volvemos a las casas, ahora como **datos**. Tenemos una tabla (simulada) de 120 viviendas. Para cada una se registró el área del primer piso, el área del segundo piso (0 si no tiene), el área total y el número de habitaciones. Guardamos la tabla en una matriz $X$ de $120\times 4$: cada **fila** es una vivienda y cada **columna**, una variable.

Un **modelo lineal de precios** asigna un peso a cada variable, $\mathbf c=(c_1,c_2,c_3,c_4)$, y estima el precio de una vivienda como

$$
\text{precio estimado}=c_1\,(\text{área piso 1})+c_2\,(\text{área piso 2})+c_3\,(\text{área total})+c_4\,(\text{habitaciones}).
$$

Para las 120 viviendas a la vez, $\hat{\mathbf p}=X\mathbf c$. Es decir, la tabla define una transformación lineal

$$
T_X:\mathbb{R}^4\to\mathbb{R}^{120},\qquad \mathbf c\longmapsto X\mathbf c,
$$

que convierte un vector de **pesos** en un vector de **120 precios estimados**. (Ojo: aquí el dominio es el espacio de los pesos, no el de las viviendas.)

In [ ]:
rng = np.random.default_rng(2026)
N = 120
piso1 = np.round(rng.uniform(45, 95, N), 1)                          # m²
piso2 = np.where(rng.random(N) < 0.65, np.round(rng.uniform(30, 80, N), 1), 0.0)
total = piso1 + piso2
habitaciones = np.clip(np.round(total / 40 + rng.normal(0, 0.7, N)), 1, 6)
precio = 3.2 * total + 12 * habitaciones + rng.normal(0, 15, N)     # millones de pesos (datos simulados)

X = np.column_stack([piso1, piso2, total, habitaciones])
nombres = ["área piso 1", "área piso 2", "área total", "habitaciones"]
print("X tiene tamaño", X.shape, "\n")
print(f"{'vivienda':>8} | " + " | ".join(f"{s:>12}" for s in nombres) + f" | {'precio':>8}")
print("-" * 82)
for i in range(8):
    print(f"{i + 1:>8} | " + " | ".join(f"{x:>12.1f}" for x in X[i]) + f" | {precio[i]:>8.1f}")
print("     ... | (112 viviendas más)")

🔮 Mira las columnas de la tabla. ¿Notas alguna relación entre ellas? ¿Qué dimensión crees que tendrá el núcleo de $T_X$?

In [ ]:
analizar(X, "T_X", mostrar_matriz=False)
k_X = base_nucleo(X)[:, 0]
print("\nPara las primeras 5 viviendas, (fila de X) · k =", X[:5] @ k_X)

El núcleo está generado por $\mathbf k=(-1,-1,1,0)$. Que $X\mathbf k=\mathbf 0$ dice, fila por fila, que para **toda** vivienda

$$
-(\text{área piso 1})-(\text{área piso 2})+(\text{área total})=0 .
$$

La columna "área total" es **redundante**: es la suma de otras dos. Veamos dos consecuencias.

**Consecuencia 1: los pesos no son únicos.** Por la propiedad clave, si $\mathbf c_b-\mathbf c_a\in\operatorname{Nu}(T_X)$, entonces $X\mathbf c_a=X\mathbf c_b$. Tres analistas pueden proponer pesos muy distintos que producen **exactamente** las mismas 120 predicciones.

In [ ]:
c_a = np.array([0.0, 0.0, 3.2, 12.0])        # analista A: solo usa el área total
c_b = c_a - 3.2 * k_X                        # analista B: solo usa las áreas de cada piso
c_c = c_a + 5.0 * k_X                        # analista C: pesos negativos para cada piso (!)
for nombre, c in [("A", c_a), ("B", c_b), ("C", c_c)]:
    print(f"Pesos del analista {nombre}: {fmt(c):<26}  precio estimado de la vivienda 1: {(X @ c)[0]:.2f}")
print("\nMáxima diferencia entre las predicciones de A y C:", np.abs(X @ c_a - X @ c_c).max())

fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
ancho = 0.26
for j, (nombre, c, col) in enumerate([("A", c_a, "#2e86c1"), ("B", c_b, "#e67e22"), ("C", c_c, "#8e44ad")]):
    axs[0].bar(np.arange(4) + (j - 1) * ancho, c, ancho, label=f"analista {nombre}", color=col)
axs[0].axhline(0, color="k", lw=0.8)
axs[0].set_xticks(range(4)); axs[0].set_xticklabels(nombres)
axs[0].set_title("Tres vectores de pesos muy distintos…"); axs[0].legend()
axs[1].scatter(X @ c_a, X @ c_b, s=18, color="#e67e22", label="B contra A")
axs[1].scatter(X @ c_a, X @ c_c, s=6, color="#8e44ad", label="C contra A")
lim = [(X @ c_a).min() - 20, (X @ c_a).max() + 20]
axs[1].plot(lim, lim, "k--", lw=0.8, label="y = x")
axs[1].set_xlabel("precio estimado por A (millones)"); axs[1].set_ylabel("precio estimado por B y por C")
axs[1].set_title("…producen las mismas 120 predicciones"); axs[1].legend()
plt.tight_layout(); plt.show()

**Consecuencia 2: los datos viven en un plano.** Si dibujamos cada vivienda como un punto (área piso 1, área piso 2, área total) de $\mathbb{R}^3$, todos los puntos cumplen $-x-y+z=0$: están en un plano que pasa por el origen, y el vector $(-1,-1,1)$ (las tres primeras componentes de $\mathbf k$) es perpendicular a ese plano. Las filas de $X$ son perpendiculares a los vectores del núcleo.

In [ ]:
P3 = np.vstack([piso1, piso2, total])                  # cada vivienda como un punto de R^3
normal = np.array([-1.0, -1.0, 1.0]) / np.sqrt(3)       # las tres primeras componentes de k, de longitud 1
p0 = np.array([60.0, 40.0, 100.0])                      # un punto del plano, para apoyar la flecha
gx, gy = np.meshgrid(np.linspace(0, 100, 2), np.linspace(0, 85, 2))
fig = plt.figure(figsize=(12, 5.4))
for k_v, (elev, azim, t) in enumerate([(20, -60, "Las viviendas y el plano  total = piso 1 + piso 2"),
                                       (0, -45, "Vista de canto: la nube de puntos es plana")]):
    ax = fig.add_subplot(1, 2, k_v + 1, projection="3d")
    ax.set_proj_type("ortho")
    ax.scatter(*P3, s=12, color="#2e86c1", depthshade=False)
    ax.plot_surface(gx, gy, gx + gy, color="#2e86c1", alpha=0.12, shade=False)
    ax.quiver(*p0, *(40 * normal), color=COLOR_NUCLEO, linewidth=2.5, arrow_length_ratio=0.2)
    ax.text(*(p0 + 46 * normal), "(−1, −1, 1)", color=COLOR_NUCLEO, fontweight="bold")
    ax.set_xlim(0, 100); ax.set_ylim(0, 85); ax.set_zlim(0, 185)
    ax.set_box_aspect((100, 85, 185))                  # la misma escala en los tres ejes
    ax.set_xlabel("área piso 1"); ax.set_ylabel("área piso 2"); ax.set_zlabel("área total")
    for eje in (ax.xaxis, ax.yaxis, ax.zaxis):
        eje.set_major_locator(MaxNLocator(4))
    if elev == 0:                                      # de canto, los números de x e y se amontonan
        ax.set_xticklabels([]); ax.set_yticklabels([])
    ax.view_init(elev=elev, azim=azim); ax.set_title(t)
plt.tight_layout(); plt.show()

**La solución: quitar la columna redundante.** Si eliminamos "área total", la nueva tabla $X_2$ ($120\times 3$) define $T_{X_2}:\mathbb{R}^3\to\mathbb{R}^{120}$.

🔮 ¿Cuál será ahora su núcleo?

In [ ]:
X2 = X[:, [0, 1, 3]]                                    # piso 1, piso 2, habitaciones
analizar(X2, "T_X2", mostrar_matriz=False)

### 🤔 Preguntas para pensar (2.2)

a) Explica con palabras qué significa que $(-1,-1,1,0)$ esté en el núcleo de $T_X$.

b) El analista A dice: "el área de cada piso no influye en el precio; solo importa el área total". El analista B dice: "cada m² del primer piso suma 3.2 millones al precio". ¿Quién tiene razón? ¿Pueden estos datos decidirlo?

c) Aplica el teorema de la dimensión a $T_X$. ¿Por qué se suma 4 y no 120?

d) $\operatorname{Im}(T_X)$ es un subespacio de $\mathbb{R}^{120}$. ¿Cuál es su dimensión? ¿Qué significa que el vector de precios reales $\mathbf p$ (casi seguro) **no** esté en la imagen?

e) Si en lugar de "área total" eliminamos la columna "habitaciones", ¿desaparece el problema? Compruébalo con `analizar(X[:, [0, 1, 2]], mostrar_matriz=False)`.

f) Después de quitar la columna redundante, ¿por qué los pesos del modelo ya quedan determinados de forma única (si existen)?

<details>
<summary><b>▶ Ver respuestas sugeridas</b> (inténtalo antes de abrir)</summary>

a) Que $-(\text{piso 1})-(\text{piso 2})+(\text{total})=0$ para todas las viviendas: una columna es combinación lineal de otras. En términos de pesos: sumar $t\,\mathbf k$ a cualquier vector de pesos no cambia ninguna predicción.

b) Con estos datos no se puede decidir: los dos modelos producen exactamente las mismas predicciones, porque sus pesos difieren en un vector del núcleo. La pregunta "¿cuánto aporta el piso 1 por separado del total?" no tiene respuesta con estas variables: los datos no pueden separar esos efectos. (En estadística esto se llama **multicolinealidad**: los coeficientes no son identificables.)

c) $\nu+\rho=1+3=4$. El teorema usa la dimensión del **dominio** de $T_X$, que es el espacio de los pesos, $\mathbb{R}^4$. El 120 es la dimensión del codominio.

d) $\rho=3$: la imagen es un subespacio de dimensión 3 dentro de $\mathbb{R}^{120}$, una especie de "plano" muy delgado. Si $\mathbf p\notin\operatorname{Im}(T_X)$, ningún vector de pesos reproduce exactamente los 120 precios (por el ruido de los datos). Lo mejor que se puede hacer es buscar el vector de la imagen más cercano a $\mathbf p$: su **proyección ortogonal** sobre la imagen. Eso es la regresión lineal (ver "Para ir más allá").

e) No. La redundancia está entre piso 1, piso 2 y total; si quitas "habitaciones", el núcleo sigue generado por $(-1,-1,1)$. Hay que quitar una de las tres columnas involucradas.

f) Porque $\operatorname{Nu}(T_{X_2})=\{\mathbf 0\}$: si $X_2\mathbf c_1=X_2\mathbf c_2$, entonces $\mathbf c_1-\mathbf c_2\in\operatorname{Nu}=\{\mathbf 0\}$, es decir, $\mathbf c_1=\mathbf c_2$. $T_{X_2}$ es inyectiva.

</details>

### Para ir más allá (opcional)

**La regresión lineal es una proyección.** El método de mínimos cuadrados (`np.linalg.lstsq`) busca los pesos $\hat{\mathbf c}$ tales que $X_2\hat{\mathbf c}$ es el vector de $\operatorname{Im}(T_{X_2})$ más cercano a los precios reales $\mathbf p$. Ese vector es la **proyección ortogonal** de $\mathbf p$ sobre la imagen: el error $\mathbf r=\mathbf p-X_2\hat{\mathbf c}$ es perpendicular a todas las columnas de $X_2$, igual que los rayos de la proyección ortogonal de la sección 1.7.4 eran perpendiculares al plano.

In [ ]:
c_hat, *_ = np.linalg.lstsq(X2, precio, rcond=None)
p_hat = X2 @ c_hat                                       # la proyección de p sobre Im(T_X2)
r = precio - p_hat                                       # el error de la proyección
print("Pesos estimados (piso 1, piso 2, habitaciones):", c_hat)
print("Productos punto del error con cada columna de X2:", X2.T @ r, " ← ¡prácticamente cero!")

plt.figure(figsize=(5.2, 4.6))
plt.scatter(precio, p_hat, s=14, color="#2e86c1")
lim = [precio.min() - 20, precio.max() + 20]
plt.plot(lim, lim, "k--", lw=0.8)
plt.xlabel("precio real (millones)"); plt.ylabel("precio estimado = proyección")
plt.title("Precio real contra su proyección sobre Im(T_X2)"); plt.tight_layout(); plt.show()

**¿Y si la redundancia no es exacta?** En la vida real el área total se mide por separado y trae pequeños errores (por ejemplo, de medio metro cuadrado). Entonces la columna ya no es exactamente la suma de las otras dos, $X$ tiene núcleo $\{\mathbf 0\}$… pero **casi** no. Repitamos el ajuste con tres errores de medición distintos.

In [ ]:
print("Pesos estimados con la tabla completa (piso 1, piso 2, total, habitaciones):")
for semilla in [1, 2, 3]:
    total_medido = total + np.random.default_rng(semilla).normal(0, 0.5, N)   # error de medición pequeño
    X_med = np.column_stack([piso1, piso2, total_medido, habitaciones])
    c_med, *_ = np.linalg.lstsq(X_med, precio, rcond=None)
    print(f"  medición {semilla}:  pesos = {fmt(c_med):<34} nulidad = {base_nucleo(X_med).shape[1]}"
          f"   |X k| / |k| = {np.linalg.norm(X_med @ k_X) / np.linalg.norm(k_X):.2f}")
print("  (compara |X k| / |k| con el tamaño típico de una columna de X:", f"{np.linalg.norm(X[:, 2]):.0f})")

🤔 **Para pensar.** Ahora la nulidad es 0, pero los pesos cambian mucho de una medición a otra, aunque las predicciones casi no cambian. ¿Por qué? Piensa en qué le pasa al vector $X\mathbf k$: ya no es exactamente cero, pero es muy pequeño comparado con las columnas de $X$.

<details>
<summary><b>▶ Ver una explicación</b> (inténtalo antes de abrir)</summary>

La dirección $\mathbf k=(-1,-1,1,0)$ está "casi" en el núcleo: $X\mathbf k$ es diminuto, así que sumar múltiplos de $\mathbf k$ a los pesos casi no cambia las predicciones. Por eso los datos casi no "ven" esa dirección, y lo que decide los pesos a lo largo de ella es el ruido. Moraleja: con datos reales un núcleo puede ser solo aproximado, y aun así causa problemas. Por eso conviene eliminar variables redundantes (o usar técnicas como la regularización, que verás en cursos de estadística o de aprendizaje automático).

</details>

---
# Síntesis

| | Casa → planta, sombra, foto | Color → gris | Pesos → predicciones |
|---|---|---|---|
| **Transformación** | $\mathbb{R}^3\to\mathbb{R}^3$ (o $\mathbb{R}^2$) | $\mathbb{R}^3\to\mathbb{R}$ | $\mathbb{R}^4\to\mathbb{R}^{120}$ |
| **Núcleo** | la dirección en que se aplasta (vertical, rayos del sol, línea de visión) | los cambios de color que no cambian el gris | los cambios de pesos que no cambian ninguna predicción |
| **Imagen** | el plano (o la recta) donde cae la figura | todos los tonos de gris | las predicciones posibles (un subespacio de dimensión 3) |
| **Lo que se pierde** | la altura o la profundidad | el tono; solo queda el brillo | la posibilidad de separar el efecto de cada variable |

**Tres ideas para llevarse:**

1. Una transformación lineal queda determinada por lo que les hace a los vectores canónicos: **las columnas de $A$ son $T(\mathbf e_j)$**.
2. $T(\mathbf u)=T(\mathbf v)\iff\mathbf u-\mathbf v\in\operatorname{Nu}(T)$: **el núcleo es exactamente la información que se pierde.** Si el núcleo es $\{\mathbf 0\}$, no se pierde nada y la transformación es inyectiva.
3. $\nu(T)+\rho(T)=n$: **las dimensiones del dominio se reparten** entre lo que se pierde (núcleo) y lo que sobrevive (imagen).

### 🤔 Preguntas de cierre

a) Explícale a un compañero, sin fórmulas, qué es el núcleo y qué es la imagen de una transformación. Usa un ejemplo de este cuaderno.

b) Completa: "Si dos entradas distintas producen la misma salida, entonces su diferencia ______".

c) Piensa en un ejemplo de tu carrera o de tu vida diaria donde una transformación lineal pierda información. ¿Cuál sería su núcleo?

d) Sea $T:\mathbb{R}^5\to\mathbb{R}^3$ lineal. ¿Cuál es el menor valor posible de su nulidad? ¿Puede ser inyectiva? ¿Puede ser sobreyectiva?

e) ¿Qué idea de este cuaderno te costó más? Escribe la pregunta que le harías a tu profesor en la próxima clase.

<details>
<summary><b>▶ Ver respuestas de b) y d)</b> (inténtalo antes de abrir)</summary>

b) …está en el núcleo de la transformación.

d) Como la imagen está dentro de $\mathbb{R}^3$, $\rho\le 3$, y por el teorema $\nu=5-\rho\ge 2$. Nunca es inyectiva (su núcleo tiene al menos dimensión 2). Sí puede ser sobreyectiva, cuando $\rho=3$ (y entonces $\nu=2$).

Las preguntas a), c) y e) son personales: compara tu respuesta con la de un compañero.

</details>

### ✅ Autoevaluación

- ☐ Puedo escribir la matriz de una transformación lineal a partir de las imágenes de $\mathbf e_1,\mathbf e_2,\mathbf e_3$.
- ☐ Puedo decidir si una función es lineal y dar un contraejemplo cuando no lo es.
- ☐ Puedo calcular bases del núcleo y de la imagen con Gauss–Jordan.
- ☐ Puedo explicar con un dibujo el núcleo y la imagen de una proyección.
- ☐ Puedo usar el teorema de la dimensión para razonar sin hacer cuentas.
- ☐ Puedo explicar qué información se pierde en un ejemplo con datos.

Si alguna casilla quedó vacía, vuelve a la sección correspondiente y repite los ✍️ **Tu turno** con otros números.